# ⚛️ CERN Beschleuniger-Schaltzentrale

## Physikalisch kohärente Teilchenstrahl-Simulation

In dieser Simulation durchläuft **dasselbe Teilchenpaket (Bunch)** den gesamten realen Beschleunigerpfad des CERN:

### 🔵 Protonenpfad
**Quelle** → **LINAC 4** ($160\text{ MeV}$) → **PSB** ($2\text{ GeV}$) → **PS** ($26\text{ GeV}$) → **SPS** ($450\text{ GeV}$) → **LHC** ($6.8\text{ TeV}$)

### 🟣 Blei-Ionen-Pfad
**Quelle** → **LINAC 3** ($4.2\text{ MeV/u}$) → **LEIR** ($72\text{ MeV/u}$) → **PS** ($5.9\text{ GeV/u}$) → **SPS** ($177\text{ GeV/u}$) → **LHC** ($2.56\text{ TeV/u}$)

### Ablauf
1. **Teilchenart wählen** (Protonen oder Blei-Ionen)
2. **Bunches injizieren**: Jeder Klick schickt ein Bunch durch die gesamte Kette. Du siehst den Punkt von der Quelle durch jeden Ring wandern.
3. **LHC auffüllen**: Der LHC braucht mindestens **6 Bunches pro Strahl** (Beam 1 im Uhrzeigersinn über TI 2, Beam 2 gegen den Uhrzeigersinn über TI 8).
4. **Energie rampen**: Alle LHC-Bunches beschleunigen synchron auf Kollisionsenergie.
5. **Kollidieren**: Erst wenn beide Strahlen gefüllt und auf Maximalenergie sind, können Kollisionen an den Detektoren (ATLAS, CMS, ALICE, LHCb) ausgelöst werden.


In [1]:
import numpy as np, matplotlib.pyplot as plt, sys, os
from IPython.display import display, HTML
sys.path.append(os.path.abspath('../scripts'))
try:
    import cern_utils as cu; print('✅ cern_utils geladen')
except: pass
print('⚙️ Physik-Engine bereit.')


✅ cern_utils geladen
⚙️ Physik-Engine bereit.


## 🎛️ Stellwerk starten
Führe die nächste Zelle aus. Dann:
1. Wähle **Protonen** oder **Blei-Ionen** (oder nutze eines der **Experiment-Presets**)
2. Klicke **Füllprotokoll (Autopilot)** für eine automatische, symmetrische Injektion beider Strahlen
3. Alternativ: Injiziere manuelle Bunches für **Beam 1** (über TI 2) und **Beam 2** (über TI 8)
4. Starte das **Ramping** und führe nach dem **Beam Squeeze** Kollisionen aus!


In [2]:
display(HTML(r'''<div id="cern-v4">
<style>
#cern-v4{background:#0d1117;color:#c9d1d9;font-family:-apple-system,'Segoe UI',Roboto,sans-serif;border-radius:16px;padding:20px;border:1px solid #30363d;max-width:1100px;margin:0 auto;user-select:none}
.cv4-hdr{display:flex;justify-content:space-between;align-items:center;border-bottom:2px solid #21262d;padding-bottom:10px;margin-bottom:14px}
.cv4-logo{font-size:20px;font-weight:800;color:#58a6ff;letter-spacing:1px}
.cv4-badge{background:rgba(88,166,255,.12);color:#58a6ff;font-size:10px;padding:3px 7px;border-radius:10px;border:1px solid rgba(88,166,255,.25);margin-left:8px}
.cv4-status{font-size:11px;color:#8b949e;display:flex;align-items:center;gap:6px}
.cv4-dot{width:8px;height:8px;border-radius:50%;background:#8b949e;display:inline-block}
.cv4-dot.on{background:#2ea44f;box-shadow:0 0 8px #2ea44f;animation:cv4p 1.5s infinite}
@keyframes cv4p{0%,100%{opacity:.6;transform:scale(.9)}50%{opacity:1;transform:scale(1.2)}}
.cv4-sel{display:flex;gap:6px;margin-bottom:14px}
.cv4-sel-tab{flex:1;padding:8px;font-size:13px;font-weight:700;text-align:center;border-radius:6px;cursor:pointer;border:1px solid #30363d;background:#161b22;transition:all .2s}
.cv4-sel-tab.act-p{background:rgba(88,166,255,.12);border-color:#58a6ff;color:#58a6ff}
.cv4-sel-tab.act-i{background:rgba(227,119,194,.12);border-color:#e377c2;color:#e377c2}
.cv4-grid{display:grid;grid-template-columns:1fr 310px;gap:20px}
@media(max-width:860px){.cv4-grid{grid-template-columns:1fr}}
.cv4-svg-wrap{background:#090d13;border-radius:12px;border:1px solid #21262d;height:500px;display:flex;align-items:center;justify-content:center;position:relative;overflow:hidden}
.cv4-panel{background:#161b22;border-radius:12px;border:1px solid #30363d;padding:14px;display:flex;flex-direction:column;gap:14px}
.cv4-ptitle{font-size:11px;text-transform:uppercase;letter-spacing:1px;color:#8b949e;border-bottom:1px solid #30363d;padding-bottom:6px;margin-bottom:6px;font-weight:700}
.cv4-btn{background:#21262d;color:#c9d1d9;border:1px solid #30363d;padding:9px 14px;border-radius:6px;cursor:pointer;font-size:12px;font-weight:600;transition:all .2s;text-align:center}
.cv4-btn:hover{background:#30363d;border-color:#8b949e}
.cv4-btn.act{background:rgba(88,166,255,.15);border-color:#58a6ff;color:#58a6ff}
.cv4-btn.act-i{background:rgba(227,119,194,.15);border-color:#e377c2;color:#e377c2}
.cv4-btn.danger{background:rgba(248,81,73,.08);border-color:rgba(248,81,73,.4);color:#f85149}
.cv4-btn.danger:hover{background:rgba(248,81,73,.18);box-shadow:0 0 10px rgba(248,81,73,.25)}
.cv4-btn.off{opacity:.3;pointer-events:none}
.cv4-fill-row{display:flex;align-items:center;gap:8px;font-size:11px}
.cv4-fill-bar{flex:1;background:#21262d;border-radius:3px;height:8px;overflow:hidden}
.cv4-fill-bar-inner{height:100%;transition:width .3s;border-radius:3px}
.cv4-fill-bar-inner.b1{background:#58a6ff}
.cv4-fill-bar-inner.b2{background:#ff7f0e}
.cv4-fill-bar-inner.b1i{background:#e377c2}
.cv4-fill-bar-inner.b2i{background:#c77dff}
.cv4-grid{display:grid;grid-template-columns:1fr 310px;gap:20px}
.cv4-ro{background:#0d1117;border-radius:5px;border:1px solid #21262d;padding:7px 10px}
.cv4-ro-l{font-size:9px;color:#8b949e;text-transform:uppercase}
.cv4-ro-v{font-size:14px;font-weight:700;color:#f0f6fc;font-family:'Courier New',monospace}
.cv4-tracker{display:flex;align-items:center;gap:4px;font-size:10px;color:#8b949e;margin-top:4px}
.cv4-tracker .step{padding:2px 6px;border-radius:3px;border:1px solid #21262d;background:#0d1117}
.cv4-tracker .step.cur{border-color:#58a6ff;color:#58a6ff;background:rgba(88,166,255,.1)}
.cv4-tracker .step.cur-i{border-color:#e377c2;color:#e377c2;background:rgba(227,119,194,.1)}
.cv4-tracker .step.done{border-color:#2ea44f;color:#2ea44f}
.cv4-tracker .arr{color:#30363d}
.svg-path{stroke:#1a1f27;stroke-width:2.5;fill:none}
.svg-path.lit{stroke:#58a6ff;filter:drop-shadow(0 0 5px rgba(88,166,255,.6))}
.svg-path.lit-i{stroke:#e377c2;filter:drop-shadow(0 0 5px rgba(227,119,194,.6))}
.svg-path.lit-b2{stroke:#ff7f0e;filter:drop-shadow(0 0 5px rgba(255,127,14,.6))}
.svg-lhc{stroke:rgba(88,166,255,.08);stroke-width:4;fill:none}
.svg-lhc.lit{stroke:rgba(88,166,255,.3)}
.svg-lhc.lit-i{stroke:rgba(227,119,194,.3)}
.svg-node{fill:#0d1117;stroke:#21262d;stroke-width:2}
.svg-node.glow{stroke:#58a6ff;fill:rgba(88,166,255,.12);filter:drop-shadow(0 0 6px rgba(88,166,255,.4))}
.svg-node.glow-i{stroke:#e377c2;fill:rgba(227,119,194,.12);filter:drop-shadow(0 0 6px rgba(227,119,194,.4))}
.svg-node.flash{stroke:#f85149;fill:rgba(248,81,73,.2);filter:drop-shadow(0 0 8px rgba(248,81,73,.6))}
.svg-lbl{font-size:9px;fill:#8b949e;font-family:monospace;text-anchor:middle}
.cv4-bottom{margin-top:20px;display:grid;grid-template-columns:1fr 1fr;gap:16px}
.cv4-evcanvas{background:#090d13;border:1px solid #21262d;border-radius:8px;width:100%;height:180px}
.cv4-histwrap{background:#090d13;border:1px solid #21262d;border-radius:8px;height:130px;padding:6px}
.cv4-dtabs{display:flex;gap:3px;margin-bottom:6px}
.cv4-dtab{flex:1;background:#21262d;border:1px solid #30363d;padding:5px;font-size:10px;color:#8b949e;border-radius:4px;cursor:pointer;text-align:center;font-weight:700}
.cv4-dtab.act{background:rgba(88,166,255,.12);border-color:#58a6ff;color:#58a6ff}
.cv4-sli-lbl{display:flex;justify-content:space-between;font-size:10px;color:#8b949e;margin-top:4px}
.cv4-sli{width:100%;background:#21262d;border-radius:3px;height:4px;outline:none;-webkit-appearance:none;margin-top:2px}
.cv4-sli::-webkit-slider-thumb{-webkit-appearance:none;appearance:none;width:12px;height:12px;border-radius:50%;background:#58a6ff;cursor:pointer;border:1px solid #30363d}
.cv4-sli::-moz-range-thumb{width:12px;height:12px;border-radius:50%;background:#58a6ff;cursor:pointer;border:1px solid #30363d}
.cv4-quench{background:rgba(248,81,73,.15);border:1px solid #f85149;color:#f85149;padding:10px;border-radius:6px;font-size:12px;font-weight:bold;text-align:center;animation:cv4blink 1s infinite}
@keyframes cv4blink{0%,100%{opacity:.5}50%{opacity:1}}
.geo-element{transition:opacity 0.3s ease, fill-opacity 0.3s ease;}
</style>

<div class="cv4-hdr">
 <div><span class="cv4-logo">⚛️ CERN CCC</span><span class="cv4-badge">Schaltzentrale v5 - Real Map</span></div>
 <div style="display:flex;align-items:center;gap:12px">
  <button class="cv4-btn act" id="btn-toggle-geo" style="padding:4px 8px;font-size:10.5px">🌐 Geo-Overlay</button>
  <div class="cv4-status"><span class="cv4-dot" id="sdot"></span><span id="stxt">OFFLINE</span></div>
 </div>
</div>

<div class="cv4-sel">
  <div class="cv4-sel-tab act-p" id="sel-p">🔵 Protonen (LINAC4 → PSB → PS → SPS → LHC)</div>
  <div class="cv4-sel-tab" id="sel-i">🟣 Blei-Ionen (LINAC3 → LEIR → PS → SPS → LHC)</div>
</div>

<div class="cv4-grid">
 <div class="cv4-svg-wrap">
  <!-- Interactive Absolute Overlay Reset Zoom Button -->
  <button class="cv4-btn off" id="btn-zoom-out" style="position:absolute;top:12px;left:12px;padding:5px 10px;font-size:10px;background:rgba(22,27,34,0.85);border-color:#30363d;z-index:10;transition:all 0.2s">🔍 Ansicht zurücksetzen</button>

  <svg id="svg" width="700" height="480" viewBox="0 0 700 480" style="background:#090d13">
   <!-- Architectural Grid for tech style -->
   <defs>
    <pattern id="arch-grid" width="30" height="30" patternUnits="userSpaceOnUse">
     <path d="M 30 0 L 0 0 0 30" fill="none" stroke="rgba(255,255,255,0.012)" stroke-width="0.5"/>
    </pattern>
   </defs>
   <rect width="100%" height="100%" fill="url(#arch-grid)" />

   <!-- GEOGRAPHICAL FEATURES (Toggleable via .geo-element) -->
   <!-- Geneva Lake (Lac Léman) in top-right -->
   <path class="geo-element" d="M 520,0 Q 560,50 620,60 T 700,75 L 700,0 Z" fill="rgba(88,166,255,0.04)" stroke="rgba(88,166,255,0.12)" stroke-width="1.5" />
   <text class="geo-element" x="610" y="30" fill="rgba(88,166,255,0.22)" font-size="8px" font-family="monospace">LAC LÉMAN (GENFER SEE)</text>

   <!-- French-Swiss Border (dashed line cutting diagonally) -->
   <path class="geo-element" d="M 0,220 L 700,120" stroke="rgba(255,255,255,0.06)" stroke-width="1.2" stroke-dasharray="6,6" />
   <text class="geo-element" x="80" y="200" fill="rgba(255,255,255,0.12)" font-size="7.5px" font-family="monospace" transform="rotate(-8, 80, 200)">STAATSGRENZE SCHWEIZ (CH) - FRANKREICH (FR)</text>

   <!-- Jura Mountain Ridge in the top-left -->
   <path class="geo-element" d="M 0,50 Q 80,80 150,50 T 250,20" stroke="rgba(255,255,255,0.04)" stroke-width="1.5" stroke-dasharray="3,6" fill="none" />
   <text class="geo-element" x="60" y="40" fill="rgba(255,255,255,0.10)" font-size="7px" font-family="monospace">JURA-GEBIRGE (FR)</text>

   <!-- Geographic Town/Site Markers -->
   <text class="geo-element" x="142" y="430" fill="rgba(255,255,255,0.18)" font-size="7.5px" font-family="monospace" text-anchor="middle">CERN Meyrin Campus (CH)</text>
   <text class="geo-element" x="430" y="90" fill="rgba(255,255,255,0.18)" font-size="7.5px" font-family="monospace" text-anchor="middle">CERN Prévessin Campus (FR)</text>
   <text class="geo-element" x="100" y="225" fill="rgba(255,255,255,0.18)" font-size="7.5px" font-family="monospace">St. Genis-Pouilly (FR)</text>
   <text class="geo-element" x="350" y="25" fill="rgba(255,255,255,0.18)" font-size="7.5px" font-family="monospace" text-anchor="middle">Ferney-Voltaire (FR)</text>
   <path class="geo-element" d="M 590,320 L 670,290" stroke="rgba(255,255,255,0.08)" stroke-width="3" />
   <text class="geo-element" x="630" y="332" fill="rgba(255,255,255,0.15)" font-size="7.5px" font-family="monospace" text-anchor="middle">Flughafen Genf (GVA)</text>

   <!-- REAL CERN TOP-VIEW ACCELERATOR STRUCTURE -->
   <!-- LINAC4: straight line injecting into PSB (cx=142, cy=385) -->
   <path id="p-linac4" d="M 30,385 L 124,385" class="svg-path"/>
   <!-- PSB ring cx=142 cy=385 r=18 -->
   <circle id="p-psb" cx="142" cy="385" r="18" class="svg-path"/>
   <!-- Transfer PSB→PS: starts at PSB exit angle toward PS, ends at PS entry -->
   <path id="p-psb-ps" d="M 157.7,376.2 Q 185,358 206.9,348.6" class="svg-path"/>

   <!-- LINAC3: straight line ending at LEIR left edge (124,275) -->
   <path id="p-linac3" d="M 30,275 L 124,275" class="svg-path"/>
   <!-- LEIR ring cx=142 cy=275 r=18 -->
   <circle id="p-leir" cx="142" cy="275" r="18" class="svg-path"/>
   <!-- Transfer LEIR→PS -->
   <path id="p-leir-ps" d="M 157.7,283.8 Q 185,300 206.9,311.4" class="svg-path"/>

   <!-- PS ring cx=242 cy=332 r=38 -->
   <circle id="p-ps" cx="242" cy="332" r="38" class="svg-path"/>
   <!-- Transfer PS→SPS -->
   <path id="p-ps-sps" d="M 265.2,301.6 Q 312,248 356.8,198.6" class="svg-path"/>

   <!-- SPS ring: medium ring located SOUTH-EAST inside the LHC, near ALICE. cx=400 cy=148 r=65 -->
   <circle id="p-sps" cx="400" cy="148" r="65" class="svg-path"/>

   <!-- TI 2: SPS → LHC injection near ALICE (Point 2, left) -->
   <path id="p-ti2" d="M 339.5,173.7 Q 255,195 170,240" class="svg-path"/>
   <!-- TI 8: SPS → LHC injection near LHCb (Point 8, right) -->
   <path id="p-ti8" d="M 453.4,187.1 Q 495,215 530,240" class="svg-path"/>

   <!-- Modulated crossover beam vacuum tubes inside the LHC arcs (Double-bore design) -->
    <!-- Beam 1 tube: starts outer at 45°, crosses in detectors -->
    <path id="lhc-pipe1" class="geo-element" d="M 530.00,240.00 L 530.27,246.30 L 530.33,252.61 L 530.15,258.93 L 529.75,265.26 L 529.12,271.58 L 528.25,277.89 L 527.16,284.17 L 525.83,290.42 L 524.26,296.62 L 522.47,302.77 L 520.44,308.86 L 518.17,314.88 L 515.68,320.81 L 512.96,326.65 L 510.01,332.38 L 506.84,338.01 L 503.45,343.51 L 499.85,348.88 L 496.05,354.10 L 492.04,359.18 L 487.83,364.10 L 483.44,368.86 L 478.86,373.44 L 474.10,377.83 L 469.18,382.04 L 464.10,386.05 L 458.88,389.85 L 453.51,393.45 L 448.01,396.84 L 442.38,400.01 L 436.65,402.96 L 430.81,405.68 L 424.88,408.17 L 418.86,410.44 L 412.77,412.47 L 406.62,414.26 L 400.42,415.83 L 394.17,417.16 L 387.89,418.25 L 381.58,419.12 L 375.26,419.75 L 368.93,420.15 L 362.61,420.33 L 356.30,420.27 L 350.00,420.00 L 343.73,419.51 L 337.50,418.80 L 331.30,417.88 L 325.16,416.75 L 319.07,415.41 L 313.04,413.88 L 307.08,412.15 L 301.19,410.23 L 295.38,408.12 L 289.65,405.82 L 284.00,403.35 L 278.45,400.70 L 272.99,397.89 L 267.64,394.90 L 262.38,391.76 L 257.23,388.46 L 252.20,385.00 L 247.27,381.39 L 242.47,377.64 L 237.78,373.74 L 233.22,369.70 L 228.78,365.53 L 224.47,361.22 L 220.30,356.78 L 216.26,352.22 L 212.36,347.53 L 208.61,342.73 L 205.00,337.80 L 201.54,332.77 L 198.24,327.62 L 195.10,322.36 L 192.11,317.01 L 189.30,311.55 L 186.65,306.00 L 184.18,300.35 L 181.88,294.62 L 179.77,288.81 L 177.85,282.92 L 176.12,276.96 L 174.59,270.93 L 173.25,264.84 L 172.12,258.70 L 171.20,252.50 L 170.49,246.27 L 170.00,240.00 L 169.73,233.70 L 169.67,227.39 L 169.85,221.07 L 170.25,214.74 L 170.88,208.42 L 171.75,202.11 L 172.84,195.83 L 174.17,189.58 L 175.74,183.38 L 177.53,177.23 L 179.56,171.14 L 181.83,165.12 L 184.32,159.19 L 187.04,153.35 L 189.99,147.62 L 193.16,141.99 L 196.55,136.49 L 200.15,131.12 L 203.95,125.90 L 207.96,120.82 L 212.17,115.90 L 216.56,111.14 L 221.14,106.56 L 225.90,102.17 L 230.82,97.96 L 235.90,93.95 L 241.12,90.15 L 246.49,86.55 L 251.99,83.16 L 257.62,79.99 L 263.35,77.04 L 269.19,74.32 L 275.12,71.83 L 281.14,69.56 L 287.23,67.53 L 293.38,65.74 L 299.58,64.17 L 305.83,62.84 L 312.11,61.75 L 318.42,60.88 L 324.74,60.25 L 331.07,59.85 L 337.39,59.67 L 343.70,59.73 L 350.00,60.00 L 356.27,60.49 L 362.50,61.20 L 368.70,62.12 L 374.84,63.25 L 380.93,64.59 L 386.96,66.12 L 392.92,67.85 L 398.81,69.77 L 404.62,71.88 L 410.35,74.18 L 416.00,76.65 L 421.55,79.30 L 427.01,82.11 L 432.36,85.10 L 437.62,88.24 L 442.77,91.54 L 447.80,95.00 L 452.73,98.61 L 457.53,102.36 L 462.22,106.26 L 466.78,110.30 L 471.22,114.47 L 475.53,118.78 L 479.70,123.22 L 483.74,127.78 L 487.64,132.47 L 491.39,137.27 L 495.00,142.20 L 498.46,147.23 L 501.76,152.38 L 504.90,157.64 L 507.89,162.99 L 510.70,168.45 L 513.35,174.00 L 515.82,179.65 L 518.12,185.38 L 520.23,191.19 L 522.15,197.08 L 523.88,203.04 L 525.41,209.07 L 526.75,215.16 L 527.88,221.30 L 528.80,227.50 L 529.51,233.73 L 530.00,240.00" stroke="rgba(88,166,255,0.22)" stroke-width="1.2" fill="none" stroke-dasharray="3,3" style="transition: opacity 0.3s;" />
    <!-- Beam 2 tube: starts inner at 45°, crosses in detectors -->
    <path id="lhc-pipe2" class="geo-element" d="M 530.00,240.00 L 529.51,246.27 L 528.80,252.50 L 527.88,258.70 L 526.75,264.84 L 525.41,270.93 L 523.88,276.96 L 522.15,282.92 L 520.23,288.81 L 518.12,294.62 L 515.82,300.35 L 513.35,306.00 L 510.70,311.55 L 507.89,317.01 L 504.90,322.36 L 501.76,327.62 L 498.46,332.77 L 495.00,337.80 L 491.39,342.73 L 487.64,347.53 L 483.74,352.22 L 479.70,356.78 L 475.53,361.22 L 471.22,365.53 L 466.78,369.70 L 462.22,373.74 L 457.53,377.64 L 452.73,381.39 L 447.80,385.00 L 442.77,388.46 L 437.62,391.76 L 432.36,394.90 L 427.01,397.89 L 421.55,400.70 L 416.00,403.35 L 410.35,405.82 L 404.62,408.12 L 398.81,410.23 L 392.92,412.15 L 386.96,413.88 L 380.93,415.41 L 374.84,416.75 L 368.70,417.88 L 362.50,418.80 L 356.27,419.51 L 350.00,420.00 L 343.70,420.27 L 337.39,420.33 L 331.07,420.15 L 324.74,419.75 L 318.42,419.12 L 312.11,418.25 L 305.83,417.16 L 299.58,415.83 L 293.38,414.26 L 287.23,412.47 L 281.14,410.44 L 275.12,408.17 L 269.19,405.68 L 263.35,402.96 L 257.62,400.01 L 251.99,396.84 L 246.49,393.45 L 241.12,389.85 L 235.90,386.05 L 230.82,382.04 L 225.90,377.83 L 221.14,373.44 L 216.56,368.86 L 212.17,364.10 L 207.96,359.18 L 203.95,354.10 L 200.15,348.88 L 196.55,343.51 L 193.16,338.01 L 189.99,332.38 L 187.04,326.65 L 184.32,320.81 L 181.83,314.88 L 179.56,308.86 L 177.53,302.77 L 175.74,296.62 L 174.17,290.42 L 172.84,284.17 L 171.75,277.89 L 170.88,271.58 L 170.25,265.26 L 169.85,258.93 L 169.67,252.61 L 169.73,246.30 L 170.00,240.00 L 170.49,233.73 L 171.20,227.50 L 172.12,221.30 L 173.25,215.16 L 174.59,209.07 L 176.12,203.04 L 177.85,197.08 L 179.77,191.19 L 181.88,185.38 L 184.18,179.65 L 186.65,174.00 L 189.30,168.45 L 192.11,162.99 L 195.10,157.64 L 198.24,152.38 L 201.54,147.23 L 205.00,142.20 L 208.61,137.27 L 212.36,132.47 L 216.26,127.78 L 220.30,123.22 L 224.47,118.78 L 228.78,114.47 L 233.22,110.30 L 237.78,106.26 L 242.47,102.36 L 247.27,98.61 L 252.20,95.00 L 257.23,91.54 L 262.38,88.24 L 267.64,85.10 L 272.99,82.11 L 278.45,79.30 L 284.00,76.65 L 289.65,74.18 L 295.38,71.88 L 301.19,69.77 L 307.08,67.85 L 313.04,66.12 L 319.07,64.59 L 325.16,63.25 L 331.30,62.12 L 337.50,61.20 L 343.73,60.49 L 350.00,60.00 L 356.30,59.73 L 362.61,59.67 L 368.93,59.85 L 375.26,60.25 L 381.58,60.88 L 387.89,61.75 L 394.17,62.84 L 400.42,64.17 L 406.62,65.74 L 412.77,67.53 L 418.86,69.56 L 424.88,71.83 L 430.81,74.32 L 436.65,77.04 L 442.38,79.99 L 448.01,83.16 L 453.51,86.55 L 458.88,90.15 L 464.10,93.95 L 469.18,97.96 L 474.10,102.17 L 478.86,106.56 L 483.44,111.14 L 487.83,115.90 L 492.04,120.82 L 496.05,125.90 L 499.85,131.12 L 503.45,136.49 L 506.84,141.99 L 510.01,147.62 L 512.96,153.35 L 515.68,159.19 L 518.17,165.12 L 520.44,171.14 L 522.47,177.23 L 524.26,183.38 L 525.83,189.58 L 527.16,195.83 L 528.25,202.11 L 529.12,208.42 L 529.75,214.74 L 530.15,221.07 L 530.33,227.39 L 530.27,233.70 L 530.00,240.00" stroke="rgba(255,127,14,0.22)" stroke-width="1.2" fill="none" stroke-dasharray="3,3" style="transition: opacity 0.3s;" />

   <!-- LHC tunnel (massive average ring cx=350 cy=240 r=180) -->
   <circle id="p-lhc" cx="350" cy="240" r="180" class="svg-path svg-lhc"/>

   <!-- STYLISH ACCELERATOR DETECTORS & DETAILS -->
   <!-- RF Cavities on the LHC ring (Point 4) represented as small bright rects -->
   <rect x="340" y="415" width="20" height="10" fill="rgba(255,127,14,0.2)" stroke="#ff7f0e" stroke-width="1" />
   <rect x="340" y="55" width="20" height="10" fill="rgba(255,127,14,0.2)" stroke="#ff7f0e" stroke-width="1" />
   <text x="350" y="435" fill="rgba(255,127,14,0.5)" font-size="6px" font-family="monospace" text-anchor="middle">400 MHz RF</text>

   <!-- Quadrupole focusing triplets near the detectors -->
   <path d="M 330,420 L 370,420" stroke="#2ea44f" stroke-width="3" opacity="0.3" />
   <path d="M 330,60 L 370,60" stroke="#2ea44f" stroke-width="3" opacity="0.3" />
   <path d="M 170,220 L 170,260" stroke="#2ea44f" stroke-width="3" opacity="0.3" />
   <path d="M 530,220 L 530,260" stroke="#2ea44f" stroke-width="3" opacity="0.3" />

   <!-- Nodes / Labels -->
   <circle id="n-linac4" cx="30" cy="385" r="5" class="svg-node"/>
   <text x="30" y="405" class="svg-lbl">LINAC 4</text>
   <circle id="n-psb" cx="142" cy="385" r="7" class="svg-node"/>
   <text x="142" y="415" class="svg-lbl">PSB</text>

   <circle id="n-linac3" cx="30" cy="275" r="5" class="svg-node"/>
   <text x="30" y="258" class="svg-lbl">LINAC 3</text>
   <circle id="n-leir" cx="142" cy="275" r="7" class="svg-node"/>
   <text x="142" y="256" class="svg-lbl">LEIR</text>

   <circle id="n-ps" cx="242" cy="332" r="8" class="svg-node"/>
   <text x="242" y="383" class="svg-lbl">PS</text>
   <circle id="n-sps" cx="400" cy="148" r="10" class="svg-node"/>
   <text x="400" y="230" class="svg-lbl">SPS</text>

   <!-- LHC Detector Groups for interactive zoom -->
   <g id="grp-atlas" style="cursor:pointer">
    <circle id="d-atlas" cx="350" cy="420" r="14" class="svg-node"/>
    <text x="350" y="448" class="svg-lbl" style="fill:#e6edf3;font-weight:bold">ATLAS (IP1)</text>
   </g>
   <g id="grp-cms" style="cursor:pointer">
    <circle id="d-cms" cx="350" cy="60" r="14" class="svg-node"/>
    <text x="350" y="42" class="svg-lbl" style="fill:#e6edf3;font-weight:bold">CMS (IP5)</text>
   </g>
   <g id="grp-alice" style="cursor:pointer">
    <circle id="d-alice" cx="170" cy="240" r="12" class="svg-node"/>
    <text x="134" y="240" class="svg-lbl" style="fill:#e6edf3;font-weight:bold">ALICE (IP2)</text>
   </g>
   <g id="grp-lhcb" style="cursor:pointer">
    <circle id="d-lhcb" cx="530" cy="240" r="12" class="svg-node"/>
    <text x="567" y="240" class="svg-lbl" style="fill:#e6edf3;font-weight:bold">LHCb (IP8)</text>
   </g>

   <!-- TI labels -->
   <text x="248" y="186" class="svg-lbl" style="font-size:8px">TI 2</text>
   <text x="485" y="205" class="svg-lbl" style="font-size:8px">TI 8</text>
  </svg>
 </div>

 <div class="cv4-panel">
  <div>
   <div class="cv4-ptitle">🔬 EXPERIMENT-PRESETS (SCHNELLWAHL)</div>
   <div style="display:grid;grid-template-columns:1fr 1fr;gap:6px;margin-bottom:8px">
    <button class="cv4-btn" id="btn-pre-higgs" style="background:rgba(88,166,255,.10);border-color:#58a6ff;color:#58a6ff;font-size:9.5px;padding:6px 2px">Higgs-Suche (ATLAS/CMS)</button>
    <button class="cv4-btn" id="btn-pre-qgp" style="background:rgba(227,119,194,.10);border-color:#e377c2;color:#e377c2;font-size:9.5px;padding:6px 2px">QGP-Erzeugung (ALICE)</button>
    <button class="cv4-btn" id="btn-pre-lhcb" style="background:rgba(255,127,14,.10);border-color:#ff7f0e;color:#ff7f0e;font-size:9.5px;padding:6px 2px">CP-Verletzung (LHCb)</button>
    <button class="cv4-btn" id="btn-pre-pilot" style="background:rgba(23,190,207,.10);border-color:#17becf;color:#17becf;font-size:9.5px;padding:6px 2px">Pilot-Strahl (Testrun)</button>
   </div>
  </div>

  <div>
   <div class="cv4-ptitle">📡 INJEKTION</div>
   <div style="display:flex;flex-direction:column;gap:6px">
    <button class="cv4-btn" id="btn-speed-toggle" style="background:rgba(88,166,255,.08);border-color:rgba(88,166,255,.3);color:#58a6ff;font-size:10.5px;padding:6px 4px;margin-bottom:2px">⏱️ Modus: Zeitraffer (Schnell)</button>
    <button class="cv4-btn" id="btn-auto" style="background:rgba(46,164,79,.15);border-color:#2ea44f;color:#2ea44f">⚙️ Füllprotokoll (Autopilot)</button>
    <div style="display:flex;gap:4px">
     <button class="cv4-btn" id="btn-b1" style="flex:1;padding:6px;font-size:10px">📥 Manuell B1</button>
     <button class="cv4-btn" id="btn-b2" style="flex:1;padding:6px;font-size:10px">📥 Manuell B2</button>
    </div>
   </div>
   <div class="cv4-tracker" id="tracker">
    <span class="step" id="tr-src">Quelle</span><span class="arr">→</span>
    <span class="step" id="tr-inj">PSB</span><span class="arr">→</span>
    <span class="step" id="tr-ps">PS</span><span class="arr">→</span>
    <span class="step" id="tr-sps">SPS</span><span class="arr">→</span>
    <span class="step" id="tr-lhc">LHC</span>
   </div>
  </div>

  <div>
   <div class="cv4-ptitle">🔋 LHC FÜLLSTAND</div>
   <div class="cv4-fill-row"><span style="width:50px">B1 <span id="b1c">0</span>/6</span><div class="cv4-fill-bar"><div class="cv4-fill-bar-inner b1" id="b1bar" style="width:0%"></div></div></div>
   <div class="cv4-fill-row" style="margin-top:4px"><span style="width:50px">B2 <span id="b2c">0</span>/6</span><div class="cv4-fill-bar"><div class="cv4-fill-bar-inner b2" id="b2bar" style="width:0%"></div></div></div>
  </div>

  <div>
   <div class="cv4-ptitle">🎛️ CCC BETRIEBSPARAMETER</div>
   <div class="cv4-sli-lbl"><span>Ziel-Energie:</span><span id="lbl-energy">6.8 TeV</span></div>
   <input type="range" class="cv4-sli" id="sli-energy" min="1.0" max="7.0" step="0.2" value="6.8">

   <div class="cv4-sli-lbl" style="margin-top:8px"><span>Bunch-Intensität:</span><span id="lbl-intensity">1.15e11 p</span></div>
   <input type="range" class="cv4-sli" id="sli-intensity" min="0.1" max="1.8" step="0.05" value="1.15">

   <div class="cv4-sli-lbl" style="margin-top:8px"><span>Strahl-Fokus β* (Squeeze):</span><span id="lbl-beta">1.50 m</span></div>
   <input type="range" class="cv4-sli" id="sli-beta" min="0.3" max="1.5" step="0.1" value="1.5" disabled>

   <div class="cv4-sli-lbl" style="margin-top:8px"><span>Ramp-Rate dB/dt:</span><span id="lbl-rampspeed" style="color:#58a6ff">0.05 T/s (Sicher)</span></div>
   <input type="range" class="cv4-sli" id="sli-rampspeed" min="0.02" max="0.15" step="0.01" value="0.05">
  </div>

  <div>
   <div class="cv4-ptitle">⚡ LHC BETRIEB</div>
   <div style="display:flex;flex-direction:column;gap:6px">
    <button class="cv4-btn off" id="btn-ramp">🚀 Energie-Ramping starten</button>
    <div class="cv4-fill-row"><span style="width:50px;font-size:10px">Ramp</span><div class="cv4-fill-bar"><div class="cv4-fill-bar-inner b1" id="rbar" style="width:0%"></div></div></div>
    <button class="cv4-btn off" id="btn-squeeze" style="background:rgba(23,190,207,.15);border-color:#17becf;color:#17becf">🗜️ Beam Squeeze starten (β*)</button>
    <div style="display:flex;gap:4px">
     <button class="cv4-btn danger off" id="btn-coll" style="flex:1;font-size:10.5px;padding:9px 2px">💥 Kollision (manuell)</button>
     <button class="cv4-btn off" id="btn-autocoll" style="flex:1.2;background:rgba(248,81,73,.08);border-color:rgba(248,81,73,.4);color:#f85149;font-size:10.5px;padding:9px 2px">▶️ Auto-Datennahme</button>
    </div>
   </div>
  </div>

  <div>
   <div class="cv4-ptitle">📊 MESSWERTE</div>
   <div class="cv4-rg">
    <div class="cv4-ro"><span class="cv4-ro-l">Energie/Beam</span><span class="cv4-ro-v" id="v-e">0.00 TeV</span></div>
    <div class="cv4-ro"><span class="cv4-ro-l">Magnetfeld B</span><span class="cv4-ro-v" id="v-b">0.000 T</span></div>
    <div class="cv4-ro"><span class="cv4-ro-l">Lorentz γ</span><span class="cv4-ro-v" id="v-g">1</span></div>
    <div class="cv4-ro"><span class="cv4-ro-l">Teilchen</span><span class="cv4-ro-v" id="v-t" style="color:#58a6ff">Proton</span></div>
   </div>
  </div>
 </div>
</div>

<div class="cv4-bottom">
 <div>
  <div class="cv4-ptitle">📸 EVENT DISPLAY</div>
  <div class="cv4-dtabs">
   <div class="cv4-dtab act" id="dt-atlas">ATLAS</div>
   <div class="cv4-dtab" id="dt-cms">CMS</div>
   <div class="cv4-dtab" id="dt-alice">ALICE</div>
   <div class="cv4-dtab" id="dt-lhcb">LHCb</div>
  </div>
  <canvas class="cv4-evcanvas" id="cv-ev"></canvas>
 </div>
 <div>
  <div class="cv4-ptitle">📊 MASSENSPEKTRUM <span id="sp-info" style="float:right;font-size:10px;color:#58a6ff">Kollisionen: 0</span></div>
  <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:3px;font-size:10px;padding:0 4px">
   <span>Signifikanz: <strong id="lbl-sig" style="font-family:monospace;color:#8b949e">0.00 σ</strong></span>
   <span id="lbl-sig-status" style="font-weight:bold;color:#8b949e;font-size:9.5px">Rauschen (Kein Signal)</span>
  </div>
  <div class="cv4-fill-bar" style="height:4px;margin-bottom:6px;background:#21262d;border-radius:2px;overflow:hidden">
   <div class="cv4-fill-bar-inner" id="sig-bar" style="width:0%;background:#8b949e;height:100%;transition:all 0.2s"></div>
  </div>
  <div class="cv4-histwrap"><canvas id="cv-hist" style="width:100%;height:100%"></canvas></div>
 </div>
</div>

<script>
(function(){
const SVG_NS="http://www.w3.org/2000/svg";
const svg=document.getElementById("svg");

// ═══════════════════════════════════════════════════════════════════════════
// GEOMETRY CONFIG — all computed so path endpoints match ring entry/exit pts
// ═══════════════════════════════════════════════════════════════════════════
const R={
 PSB:{cx:142,cy:385,r:18}, LEIR:{cx:142,cy:275,r:18},
 PS:{cx:242,cy:332,r:38}, SPS:{cx:400,cy:148,r:65},
 LHC:{cx:350,cy:240,r:180}
};
// Junction angles (radians, SVG coords: 0=right, positive=CW/downward)
const J={
 PSB_ENTRY: Math.PI,    // from LINAC (left side)
 PSB_EXIT: Math.atan2(R.PS.cy-R.PSB.cy, R.PS.cx-R.PSB.cx),     // toward PS ≈-0.51
 LEIR_ENTRY: Math.PI,
 LEIR_EXIT: Math.atan2(R.PS.cy-R.LEIR.cy, R.PS.cx-R.LEIR.cx),  // toward PS ≈0.51
 PS_FROM_PSB: Math.atan2(R.PSB.cy-R.PS.cy, R.PSB.cx-R.PS.cx),   // from PSB ≈2.63
 PS_FROM_LEIR: Math.atan2(R.LEIR.cy-R.PS.cy, R.LEIR.cx-R.PS.cx),// from LEIR ≈-2.63→3.65
 PS_EXIT: Math.atan2(R.SPS.cy-R.PS.cy, R.SPS.cx-R.PS.cx),       // toward SPS ≈-0.84
 SPS_ENTRY: Math.atan2(R.PS.cy-R.SPS.cy, R.PS.cx-R.SPS.cx),     // from PS ≈2.30
 SPS_TI2: Math.atan2(240-R.SPS.cy, 170-R.SPS.cx),                // toward ALICE ≈2.77
 SPS_TI8: Math.atan2(240-R.SPS.cy, 530-R.SPS.cx),                // toward LHCb ≈0.61
 LHC_ALICE: Math.PI,   // ALICE at 180° (left)
 LHC_LHCB: 0           // LHCb at 0° (right)
};

// ═══════════════════════════════════════════════════════════════════════════
// STATE
// ═══════════════════════════════════════════════════════════════════════════
let isIon=false, injecting=false, ramped=false;
let b1Count=0, b2Count=0, collisions=0;
const NEEDED=6;
let lhcDots={b1:[],b2:[]};
let lhcSpeed=0.0030; // rad/ms at injection energy (Proton default)
let lhcAngle=0, lhcRunning=false, lhcLastT=null;
let lhcEnergy=450; // GeV
let massData=[];
let selDet="ATLAS";
let isFastMode=true; // Toggleable speed mode

function getDurations() {
  if (isFastMode) {
    return {
      linac: isIon ? 350 : 200,
      ring1: isIon ? 900 : 500,
      trToPs: isIon ? 150 : 100,
      ps: isIon ? 1000 : 600,
      trToSps: isIon ? 200 : 100,
      sps: isIon ? 800 : 500,
      ti: isIon ? 250 : 150,
      autoDelay: 150
    };
  } else {
    return {
      linac: isIon ? 1200 : 600,
      ring1: isIon ? 3400 : 1900,
      trToPs: isIon ? 300 : 150,
      ps: isIon ? 3600 : 2000,
      trToSps: isIon ? 400 : 200,
      sps: isIon ? 2700 : 1600,
      ti: isIon ? 500 : 250,
      autoDelay: 800
    };
  }
}

// CCC OPERATOR STATES
let paramEnergy = 6.8;      // Target Energy (TeV)
let paramIntensity = 1.15;  // Bunch Intensity (10^11 protons)
let paramBetaStar = 1.5;    // Beam size at IP (meters)
let paramRampSpeed = 0.05;  // Magnetic field ramp rate (T/s)
let squeezing = false;      // Squeeze in progress
let squeezed = false;       // Squeeze complete
let cryoRecovery = false;   // Cryogenic recovery active
let autoCollInterval = null; // Auto Collide loop

// ═══════════════════════════════════════════════════════════════════════════
// DOM REFERENCES
// ═══════════════════════════════════════════════════════════════════════════
const $=id=>document.getElementById(id);
const sdot=$("sdot"),stxt=$("stxt");
const btnB1=$("btn-b1"),btnB2=$("btn-b2"),btnRamp=$("btn-ramp"),btnColl=$("btn-coll"),btnAuto=$("btn-auto"),btnSqueeze=$("btn-squeeze");
const btnAutoColl=$("btn-autocoll"),btnSpeedToggle=$("btn-speed-toggle");
const b1c=$("b1c"),b2c=$("b2c"),b1bar=$("b1bar"),b2bar=$("b2bar"),rbar=$("rbar");
const vE=$("v-e"),vB=$("v-b"),vG=$("v-g"),vT=$("v-t");
const spInfo=$("sp-info");
const sliEnergy=$("sli-energy"), sliIntensity=$("sli-intensity"), sliBeta=$("sli-beta"), sliRampSpeed=$("sli-rampspeed");
const lblEnergy=$("lbl-energy"), lblIntensity=$("lbl-intensity"), lblBeta=$("lbl-beta"), lblRampSpeed=$("lbl-rampspeed");
const trSteps=["tr-src","tr-inj","tr-ps","tr-sps","tr-lhc"].map($);
const trInj=$("tr-inj");
const selP=$("sel-p"),selI=$("sel-i");

const btnToggleGeo=$("btn-toggle-geo");
const btnPreHiggs=$("btn-pre-higgs"),btnPreQgp=$("btn-pre-qgp"),btnPreLhcb=$("btn-pre-lhcb"),btnPrePilot=$("btn-pre-pilot");
const btnZoomOut=$("btn-zoom-out");
const grpAtlas=$("grp-atlas"),grpCms=$("grp-cms"),grpAlice=$("grp-alice"),grpLhcb=$("grp-lhcb");

// SVG path elements
const paths={
 linac4:$("p-linac4"), psb:$("p-psb"), psbPs:$("p-psb-ps"),
 linac3:$("p-linac3"), leir:$("p-leir"), leirPs:$("p-leir-ps"),
 ps:$("p-ps"), psSps:$("p-ps-sps"), sps:$("p-sps"),
 ti2:$("p-ti2"), ti8:$("p-ti8"), lhc:$("p-lhc")
};
const nodes={
 linac4:$("n-linac4"), psb:$("n-psb"), linac3:$("n-linac3"), leir:$("n-leir"),
 ps:$("n-ps"), sps:$("n-sps"),
 atlas:$("d-atlas"), cms:$("d-cms"), alice:$("d-alice"), lhcb:$("d-lhcb")
};

// Canvas references
const cvEv=$("cv-ev"), ctxEv=cvEv.getContext("2d");
const cvHist=$("cv-hist"), ctxHist=cvHist.getContext("2d");

// High-DPI Resolution backing store setup for super crisp Retinas
let dpr = window.devicePixelRatio || 1;
let evW = 340, evH = 180, histW = 340, histH = 130;

function resizeCanvases(){
 // Resize Event Display
 const rEv = cvEv.getBoundingClientRect();
 evW = rEv.width || 340;
 evH = rEv.height || 180;
 cvEv.width = evW * dpr;
 cvEv.height = evH * dpr;
 cvEv.style.width = evW + "px";
 cvEv.style.height = evH + "px";
 ctxEv.resetTransform ? ctxEv.resetTransform() : null;
 ctxEv.scale(dpr, dpr);
 
 // Resize Histogram
 const rHist = cvHist.getBoundingClientRect();
 histW = rHist.width || 340;
 histH = rHist.height || 130;
 cvHist.width = histW * dpr;
 cvHist.height = histH * dpr;
 cvHist.style.width = histW + "px";
 cvHist.style.height = histH + "px";
 ctxHist.resetTransform ? ctxHist.resetTransform() : null;
 ctxHist.scale(dpr, dpr);
}

["atlas","cms","alice","lhcb"].forEach(d=>{
 $("dt-"+d).addEventListener("click",()=>{
  document.querySelectorAll(".cv4-dtab").forEach(t=>t.classList.remove("act"));
  $("dt-"+d).classList.add("act");
  selDet=d.toUpperCase();
  drawDetBg();
 });
});

selP.addEventListener("click",()=>{ if(injecting)return; setMode(false); });
selI.addEventListener("click",()=>{ if(injecting)return; setMode(true); });

function setMode(ion){
 if(isIon===ion && b1Count===0 && b2Count===0) return;
 isIon=ion;
 selP.className="cv4-sel-tab"+(ion?"":" act-p");
 selI.className="cv4-sel-tab"+(ion?" act-i":"");
 vT.innerText=ion?"Pb⁸²⁺":"Proton";
 vT.style.color=ion?"#e377c2":"#58a6ff";
 trInj.innerText=ion?"LEIR":"PSB";
 b1bar.className="cv4-fill-bar-inner "+(ion?"b1i":"b1");
 b2bar.className="cv4-fill-bar-inner "+(ion?"b2i":"b2");
 if(ion){
  document.querySelectorAll(".cv4-dtab").forEach(t=>t.classList.remove("act"));
  $("dt-alice").classList.add("act"); selDet="ALICE";
 }
 resetLHC();
 drawDetBg(); drawHist();
}

function resetLHC(){
 resetFlag = true;
 autopilotActive = false;
 stopAutoCollide();
 document.querySelectorAll(".traveling-dot").forEach(d=>d.remove());
 btnB1.classList.remove("off"); btnB2.classList.remove("off"); btnAuto.classList.remove("off");
 injecting = false;
 lhcDots.b1.forEach(d=>d.el.remove()); lhcDots.b2.forEach(d=>d.el.remove());
 lhcDots={b1:[],b2:[]}; b1Count=0; b2Count=0; collisions=0; massData=[];
 ramped=false; squeezed=false; squeezing=false;
 lhcEnergy=isIon?177:450; lhcSpeed=isIon?0.0019:0.0030;
 paramBetaStar=1.5;
 b1c.innerText="0"; b2c.innerText="0"; b1bar.style.width="0%"; b2bar.style.width="0%";
 rbar.style.width="0%"; spInfo.innerText="Kollisionen: 0";
 btnRamp.classList.add("off"); btnSqueeze.classList.add("off"); btnColl.classList.add("off");
 btnAutoColl.classList.add("off");
 sliEnergy.disabled = false; sliIntensity.disabled = false; sliBeta.value = 1.5; sliBeta.disabled = true; sliRampSpeed.disabled = false;
 lblBeta.innerText = "1.50 m";
 updateReadouts();
 Object.values(paths).forEach(p=>{p.classList.remove("lit","lit-i","lit-b2")});
 Object.values(nodes).forEach(n=>{n.classList.remove("glow","glow-i","flash")});
 paths.lhc.classList.remove("lit","lit-i");
 setStatus("BEREIT","on");
}

async function moveAlongPath(dot, pathEl, dur){
 return new Promise(res=>{
  const len=pathEl.getTotalLength();
  let t0=null;
  function step(ts){
   if(!t0) t0=ts;
   let p=Math.min((ts-t0)/dur,1);
   let pt=pathEl.getPointAtLength(p*len);
   dot.setAttribute("cx",pt.x); dot.setAttribute("cy",pt.y);
   p<1 ? requestAnimationFrame(step) : res();
  }
  requestAnimationFrame(step);
 });
}

async function orbitRing(dot, ring, entryA, exitA, orbits, dur){
  let partial=((exitA-entryA)%(2*Math.PI)+2*Math.PI)%(2*Math.PI);
  let totalA=orbits*2*Math.PI+partial;
  return new Promise(res=>{
   let t0=null;
   function step(ts){
    if(!t0) t0=ts;
    let p=Math.min((ts-t0)/dur,1);
    let a=entryA+p*totalA;
    dot.setAttribute("cx",ring.cx+ring.r*Math.cos(a));
    dot.setAttribute("cy",ring.cy+ring.r*Math.sin(a));
    p<1 ? requestAnimationFrame(step) : res();
   }
   requestAnimationFrame(step);
  });
}

async function injectBunch(beam){
 if(injecting) return;
 injecting=true;
 resetFlag = false;
 btnB1.classList.add("off"); btnB2.classList.add("off");

 const ion=isIon;
 const litCls=ion?"lit-i":"lit";
 const glowCls=ion?"glow-i":"glow";
 const dotColor=ion?"#e377c2":"#58a6ff";
 const dotColorB2=ion?"#c77dff":"#ff7f0e";
 const color=beam===1?dotColor:dotColorB2;

 const dot=document.createElementNS(SVG_NS,"circle");
 dot.setAttribute("class", "traveling-dot");
 dot.setAttribute("r","4");
 dot.setAttribute("fill",color);
 dot.style.filter="drop-shadow(0 0 4px "+color+")";
 svg.appendChild(dot);

 function lightPath(el){el.classList.add(litCls)}
 function lightNode(el){el.classList.add(glowCls)}
 function dimPath(el){el.classList.remove(litCls,"lit","lit-i","lit-b2")}
 function dimNode(el){el.classList.remove(glowCls,"glow","glow-i")}
 function setTracker(idx){
  trSteps.forEach((s,i)=>{
   s.classList.remove("cur","cur-i","done");
   if(i<idx) s.classList.add("done");
   else if(i===idx) s.classList.add(ion?"cur-i":"cur");
  });
 }

 setTracker(0);
 const linacPath=ion?paths.linac3:paths.linac4;
 const linacNode=ion?nodes.linac3:nodes.linac4;
 lightNode(linacNode); lightPath(linacPath);
 setStatus(ion?"LINAC 3 FEUERT":"LINAC 4 FEUERT","on");
 await moveAlongPath(dot, linacPath, getDurations().linac);
 if (resetFlag) { dot.remove(); return; }
 dimNode(linacNode);

 setTracker(1);
 const ring1=ion?R.LEIR:R.PSB;
 const ring1Path=ion?paths.leir:paths.psb;
 const ring1Node=ion?nodes.leir:nodes.psb;
 const ring1Entry=ion?J.LEIR_ENTRY:J.PSB_ENTRY;
 const ring1Exit=ion?J.LEIR_EXIT:J.PSB_EXIT;
 lightPath(ring1Path); lightNode(ring1Node);
 setStatus(ion?"LEIR: Beschleunigung auf 72 MeV/u":"PSB: Beschleunigung auf 2 GeV","on");
 await orbitRing(dot, ring1, ring1Entry, ring1Exit, 3, getDurations().ring1);
 if (resetFlag) { dot.remove(); return; }
 dimPath(ring1Path); dimPath(linacPath);

 const trToPs=ion?paths.leirPs:paths.psbPs;
 lightPath(trToPs);
 await moveAlongPath(dot, trToPs, getDurations().trToPs);
 if (resetFlag) { dot.remove(); return; }
 dimPath(trToPs); dimNode(ring1Node);

 setTracker(2);
 const psEntry=ion?J.PS_FROM_LEIR:J.PS_FROM_PSB;
 lightPath(paths.ps); lightNode(nodes.ps);
 setStatus(ion?"PS: Beschleunigung auf 5.9 GeV/u":"PS: Beschleunigung auf 26 GeV","on");
 await orbitRing(dot, R.PS, psEntry, J.PS_EXIT, 3, getDurations().ps);
 if (resetFlag) { dot.remove(); return; }
 dimPath(paths.ps);

 lightPath(paths.psSps);
 await moveAlongPath(dot, paths.psSps, getDurations().trToSps);
 if (resetFlag) { dot.remove(); return; }
 dimPath(paths.psSps); dimNode(nodes.ps);

 setTracker(3);
 lightPath(paths.sps); lightNode(nodes.sps);
 const spsExit=beam===1?J.SPS_TI2:J.SPS_TI8;
 setStatus(ion?"SPS: Beschleunigung auf 177 GeV/u":"SPS: Beschleunigung auf 450 GeV","on");
 await orbitRing(dot, R.SPS, J.SPS_ENTRY, spsExit, 2, getDurations().sps);
 if (resetFlag) { dot.remove(); return; }
 dimPath(paths.sps);

 const tiPath=beam===1?paths.ti2:paths.ti8;
 lightPath(tiPath);
 setStatus(beam===1?"Transfer via TI 2 zum LHC":"Transfer via TI 8 zum LHC","on");
 await moveAlongPath(dot, tiPath, getDurations().ti);
 if (resetFlag) { dot.remove(); return; }
 dimPath(tiPath); dimNode(nodes.sps);

 setTracker(4);
 dot.remove();
 addPermanentDot(beam);
 if(beam===1){ b1Count++; b1c.innerText=b1Count; b1bar.style.width=(b1Count/NEEDED*100)+"%"; }
 else { b2Count++; b2c.innerText=b2Count; b2bar.style.width=(b2Count/NEEDED*100)+"%"; }
 paths.lhc.classList.add(ion?"lit-i":"lit");
 if(b1Count>=NEEDED && b2Count>=NEEDED && !ramped){
  btnRamp.classList.remove("off");
  setStatus("LHC GEFÜLLT — Ramping möglich!","on");
 } else {
  setStatus("LHC B1:"+b1Count+"/"+NEEDED+" B2:"+b2Count+"/"+NEEDED,"on");
 }
 injecting=false;
 btnB1.classList.remove("off"); btnB2.classList.remove("off");
 trSteps.forEach(s=>s.classList.remove("cur","cur-i","done"));
}

btnRamp.addEventListener("click",async()=>{
 if(ramped||injecting||cryoRecovery) return;
 btnRamp.classList.add("off"); btnB1.classList.add("off"); btnB2.classList.add("off"); btnAuto.classList.add("off");
 sliEnergy.disabled = true; sliIntensity.disabled = true; sliRampSpeed.disabled = true;
 setStatus("RAMPING MAGNETFELD & ENERGIE...","on");
 const startE=isIon?177:450;
 const targetE=paramEnergy*1000;
 const startSpeed=isIon?0.0019:0.0030;
 const targetSpeed=(isIon?0.0042:0.0070)*(paramEnergy/(isIon?2.5:6.8));
 const dur = 200 / paramRampSpeed;
 let t0=null;
 let quenched = false;
 await new Promise(res=>{
  function step(ts){
   if(!t0) t0=ts;
   let p=Math.min((ts-t0)/dur,1);
   if(paramRampSpeed > 0.10 && p > 0.40) { quenched = true; res(); return; }
   lhcEnergy=startE+p*(targetE-startE); lhcSpeed=startSpeed+p*(targetSpeed-startSpeed);
   rbar.style.width=(p*100)+"%"; updateReadouts();
   p<1 ? requestAnimationFrame(step) : res();
  }
  requestAnimationFrame(step);
 });
 if(quenched) { triggerQuench(); return; }
 ramped=true; btnSqueeze.classList.remove("off"); sliBeta.disabled = false;
 setStatus("RAMPING BEENDET — Squeeze-Phase einleiten!","on");
});

function triggerQuench(){
 cryoRecovery = true; stopAutoCollide();
 setStatus("💥 MAGNET-QUENCH DETEKTIERT! T > 1.9 K - Strahl gedumpt!", "danger");
 sdot.className = "cv4-dot flash";
 svg.style.transition = "filter 0.5s";
 svg.style.filter = "sepia(1) saturate(3) hue-rotate(320deg)";
 let secLeft = 5;
 function cryoTick(){
  if(secLeft > 0){ setStatus(`💥 MAGNET-QUENCH! Helium-Kühlung läuft... (${secLeft}s)`, "danger"); secLeft--; setTimeout(cryoTick, 1000); }
  else { svg.style.filter = "none"; cryoRecovery = false; resetLHC(); setStatus("KÜHLUNG ERFOLGREICH — LHC BEREIT", "on"); }
 }
 cryoTick();
}

btnSqueeze.addEventListener("click",async()=>{
 if(!ramped||squeezed||squeezing||cryoRecovery) return;
 squeezing = true; btnSqueeze.classList.add("off"); sliBeta.disabled = true;
 setStatus("🗜️ BEAM SQUEEZE: Fokussiere Strahlen an den IPs...","on");
 let t0 = null; const dur = 2000; const targetBeta = parseFloat(sliBeta.value);
 await new Promise(res=>{
  function step(ts){
   if(!t0) t0=ts;
   let p=Math.min((ts-t0)/dur,1);
   paramBetaStar = 1.5 - p * (1.5 - targetBeta);
   lblBeta.innerText = paramBetaStar.toFixed(2) + " m";
   p<1 ? requestAnimationFrame(step) : res();
  }
  requestAnimationFrame(step);
 });
 squeezing = false; squeezed = true; btnColl.classList.remove("off"); btnAutoColl.classList.remove("off");
 [nodes.atlas,nodes.cms,nodes.alice,nodes.lhcb].forEach(n=>n.classList.add("glow"));
 paths.lhc.classList.add(isIon?"lit-i":"lit");
 setStatus("STABLE BEAMS — Strahlen fokussiert! Kollisionen bereit.","on");
});

function addPermanentDot(beam){
 const key=beam===1?"b1":"b2";
 const existing=lhcDots[key].length;
 const angleOffset=existing*(2*Math.PI/NEEDED);
 const dot=document.createElementNS(SVG_NS,"circle");
 dot.setAttribute("r","3.5");
 let c=beam===1?(isIon?"#e377c2":"#58a6ff"):(isIon?"#c77dff":"#ff7f0e");
 dot.setAttribute("fill",c); dot.style.filter="drop-shadow(0 0 3px "+c+")";
 svg.appendChild(dot);
 lhcDots[key].push({el:dot,off:angleOffset});
 if(!lhcRunning) startLHCLoop();
}

function startLHCLoop(){
 lhcRunning=true; lhcLastT=null;
 function frame(ts){
  if(!lhcLastT) lhcLastT=ts;
  let dt=ts-lhcLastT; lhcLastT=ts;
  lhcAngle+=lhcSpeed*dt;
  lhcDots.b1.forEach(d=>{
   let a=lhcAngle+d.off;
   let r=180 + 5.5 * Math.sin(a * 2);
   d.el.setAttribute("cx",R.LHC.cx+r*Math.cos(a)); d.el.setAttribute("cy",R.LHC.cy+r*Math.sin(a));
  });
  lhcDots.b2.forEach(d=>{
   let a=-lhcAngle+d.off;
   let r=180 - 5.5 * Math.sin(a * 2);
   d.el.setAttribute("cx",R.LHC.cx+r*Math.cos(a)); d.el.setAttribute("cy",R.LHC.cy+r*Math.sin(a));
  });
  if(lhcRunning) requestAnimationFrame(frame);
 }
 requestAnimationFrame(frame);
}

btnColl.addEventListener("click",()=>{
 if(!ramped||!squeezed||cryoRecovery) return;
 collisions+=1; spInfo.innerText="Kollisionen: "+collisions;
 let detNode=nodes[selDet.toLowerCase()];
 if(detNode){detNode.classList.add("flash");setTimeout(()=>detNode.classList.remove("flash"),350);}
 drawCollisionEvent(); generateMassData(); drawHist();
});

function toggleAutoCollide(){
 if(autoCollInterval) stopAutoCollide(); else startAutoCollide();
}

function startAutoCollide(){
 if(!ramped || !squeezed || cryoRecovery) return;
 btnAutoColl.innerText = "⏸️ Datennahme stoppen"; btnAutoColl.classList.add("act");
 btnColl.classList.add("off");
 setStatus("DATENNAHME GESTARTET: Akkumuliere Kollisionsdaten...", "on");
 autoCollInterval = setInterval(()=>{
  if(cryoRecovery) { stopAutoCollide(); return; }
  collisions += 1;
  spInfo.innerText = "Kollisionen: " + collisions;
  let detNode=nodes[selDet.toLowerCase()];
  if(detNode){ detNode.classList.add("flash"); setTimeout(()=>detNode.classList.remove("flash"), 75); }
  drawCollisionEvent(); generateMassData(); drawHist();
 }, 125);
}

function stopAutoCollide(){
 if(autoCollInterval) { clearInterval(autoCollInterval); autoCollInterval = null; }
 btnAutoColl.innerText = "▶️ Auto-Datennahme"; btnAutoColl.classList.remove("act");
 setStatus("DATENNAHME GESTOPPT", "on");
 if(ramped && squeezed && !cryoRecovery) btnColl.classList.remove("off");
}

function updateReadouts(){
 vE.innerText=(lhcEnergy/1000).toFixed(2)+" TeV"+(isIon?"/u":"");
 let B=lhcEnergy/(0.299792458*2803.95); vB.innerText=B.toFixed(3)+" T";
 let g=lhcEnergy/(isIon?193.7:0.938272); vG.innerText=Math.round(g).toLocaleString("de-DE");
}

function setStatus(txt,cls){stxt.innerText=txt;sdot.className="cv4-dot "+cls;}

function drawDetBg(){
  let w=evW,h=evH,cx=w/2,cy=h/2;
  ctxEv.clearRect(0,0,w,h);
  ctxEv.strokeStyle="#1a1f27"; ctxEv.lineWidth=1; ctxEv.strokeRect(0,0,w,h);
  ctxEv.fillStyle="#8b949e"; ctxEv.font="9px monospace"; ctxEv.fillText(selDet + " | " + (isIon?"Pb-Pb":"p-p"), 8, 16);
  if (selDet === "ATLAS") {
    ctxEv.strokeStyle="rgba(88,166,255,0.06)";
    [20, 45, 70].forEach(r=>{ ctxEv.beginPath(); ctxEv.arc(cx,cy,r,0,2*Math.PI); ctxEv.stroke(); });
    ctxEv.strokeStyle="rgba(88,166,255,0.2)"; ctxEv.lineWidth = 4;
    for(let i=0; i<8; i++) { let a = i * Math.PI / 4; ctxEv.beginPath(); ctxEv.moveTo(cx + 65*Math.cos(a), cy + 65*Math.sin(a)); ctxEv.lineTo(cx + 85*Math.cos(a), cy + 85*Math.sin(a)); ctxEv.stroke(); }
    ctxEv.lineWidth = 1; ctxEv.fillStyle="rgba(88,166,255,0.3)"; ctxEv.font="7px monospace"; ctxEv.fillText("TOROID-MAGNETE", cx - 30, cy + 82);
  } else if (selDet === "CMS") {
    ctxEv.strokeStyle="rgba(248,81,73,0.06)";
    [20, 40, 80].forEach(r=>{ ctxEv.beginPath(); ctxEv.arc(cx,cy,r,0,2*Math.PI); ctxEv.stroke(); });
    ctxEv.strokeStyle="rgba(248,81,73,0.2)"; ctxEv.lineWidth = 6;
    ctxEv.beginPath(); ctxEv.arc(cx, cy, 55, 0, 2*Math.PI); ctxEv.stroke();
    ctxEv.lineWidth = 1; ctxEv.fillStyle="rgba(248,81,73,0.3)"; ctxEv.font="7px monospace"; ctxEv.fillText("SOLENOID-SPULE", cx - 30, cy + 64);
  } else if (selDet === "ALICE") {
    ctxEv.strokeStyle="rgba(227,119,194,0.08)";
    ctxEv.beginPath(); ctxEv.arc(cx,cy,35,0,2*Math.PI); ctxEv.stroke(); ctxEv.beginPath(); ctxEv.arc(cx,cy,15,0,2*Math.PI); ctxEv.stroke();
    ctxEv.strokeStyle="rgba(227,119,194,0.2)"; ctxEv.beginPath();
    for (let i = 0; i <= 8; i++) { let a = i * Math.PI / 4; let x = cx + 80 * Math.cos(a); let y = cy + 80 * Math.sin(a); if (i === 0) ctxEv.moveTo(x, y); else ctxEv.lineTo(x, y); }
    ctxEv.stroke(); ctxEv.fillStyle="rgba(227,119,194,0.3)"; ctxEv.font="7px monospace"; ctxEv.fillText("TPC ZYLINDER", cx - 25, cy - 40); ctxEv.fillText("L3 MAGNET", cx - 20, cy + 76);
  } else if (selDet === "LHCB") {
    ctxEv.strokeStyle="rgba(255,127,14,0.12)"; ctxEv.beginPath(); ctxEv.moveTo(0, cy); ctxEv.lineTo(w, cy); ctxEv.stroke();
    let stations = [{x: 40, label: "IP"},{x: 80, label: "RICH1"},{x: 130, label: "MAGNET"},{x: 180, label: "RICH2"},{x: 230, label: "ECAL"},{x: 290, label: "MUON"}];
    stations.forEach(st => {
      ctxEv.strokeStyle = st.label === "MAGNET" ? "rgba(255,127,14,0.4)" : "rgba(255,127,14,0.15)";
      ctxEv.lineWidth = st.label === "MAGNET" ? 3 : 1;
      ctxEv.beginPath(); ctxEv.moveTo(st.x, 25); ctxEv.lineTo(st.x, h - 15); ctxEv.stroke();
      ctxEv.fillStyle = "rgba(255,127,14,0.4)"; ctxEv.font = "6.5px monospace"; ctxEv.fillText(st.label, st.x - 12, 22);
    });
    ctxEv.lineWidth = 1;
  }
}

function drawCollisionEvent(){
  drawDetBg();
  let w=evW,h=evH,cx=w/2,cy=h/2;
  if (selDet === "ATLAS") {
    for(let i=0; i<12; i++) {
      let a = Math.random()*2*Math.PI; let len = 35 + Math.random()*30;
      ctxEv.strokeStyle = Math.random() > 0.5 ? "#58a6ff" : "#ff7f0e";
      ctxEv.lineWidth = 0.8; ctxEv.beginPath(); ctxEv.moveTo(cx, cy); ctxEv.lineTo(cx+len*Math.cos(a), cy+len*Math.sin(a)); ctxEv.stroke();
    }
    for (let i = 0; i < 2; i++) {
      let a = (i === 0 ? 0.35 : 3.4) + (Math.random() - 0.5) * 0.1;
      ctxEv.strokeStyle = "#2ea44f"; ctxEv.lineWidth = 2;
      ctxEv.beginPath(); ctxEv.moveTo(cx, cy); ctxEv.lineTo(cx + 88*Math.cos(a), cy + 88*Math.sin(a)); ctxEv.stroke();
      ctxEv.fillStyle = "#2ea44f"; ctxEv.beginPath(); ctxEv.arc(cx + 88*Math.cos(a), cy + 88*Math.sin(a), 3.5, 0, 2*Math.PI); ctxEv.fill();
    }
    ctxEv.fillStyle = "#2ea44f"; ctxEv.font = "8px sans-serif"; ctxEv.fillText("Müon-Spur (μ)", cx + 45, cy + 30);
  } else if (selDet === "CMS") {
    [1.1, 4.3].forEach(ja => {
      for (let i = 0; i < 8; i++) {
        let a = ja + (Math.random() - 0.5) * 0.18; let len = 40 + Math.random() * 38;
        ctxEv.strokeStyle = "#ff7f0e"; ctxEv.lineWidth = 0.9;
        ctxEv.beginPath(); ctxEv.moveTo(cx, cy); ctxEv.lineTo(cx + len*Math.cos(a), cy + len*Math.sin(a)); ctxEv.stroke();
      }
    });
    let ea = 2.7; ctxEv.strokeStyle = "#58a6ff"; ctxEv.lineWidth = 1.8; ctxEv.beginPath();
    ctxEv.moveTo(cx, cy); ctxEv.quadraticCurveTo(cx + 40*Math.cos(ea+0.4), cy + 40*Math.sin(ea+0.4), cx + 78*Math.cos(ea), cy + 78*Math.sin(ea));
    ctxEv.stroke(); ctxEv.fillStyle = "#58a6ff"; ctxEv.beginPath(); ctxEv.arc(cx + 78*Math.cos(ea), cy + 78*Math.sin(ea), 3, 0, 2*Math.PI); ctxEv.fill();
    ctxEv.fillStyle = "#ff7f0e"; ctxEv.font = "8px sans-serif"; ctxEv.fillText("Hadron-Jets", cx + 32, cy - 25);
  } else if (selDet === "ALICE") {
    let nTracks = isIon ? 120 : 40;
    for(let i=0; i<nTracks; i++) {
      let a = Math.random()*2*Math.PI;
      let len = 15 + Math.random()*58;
      let curve = a + (Math.random() > 0.5 ? 0.22 : -0.22) * (len/70);
      let tx = cx + len*Math.cos(a), ty = cy + len*Math.sin(a);
      let cpx = cx + (len/2)*Math.cos(curve), cpy = cy + (len/2)*Math.sin(curve);
      ctxEv.strokeStyle = isIon ? "rgba(227,119,194,0.6)" : "rgba(88,166,255,0.6)";
      ctxEv.lineWidth = 0.7;
      ctxEv.beginPath(); ctxEv.moveTo(cx, cy); ctxEv.quadraticCurveTo(cpx, cpy, tx, ty); ctxEv.stroke();
    }
    ctxEv.fillStyle = "rgba(227,119,194,0.8)"; ctxEv.font = "8px sans-serif";
    ctxEv.fillText("TPC-Spuren: n = " + nTracks, cx - 40, cy - 4);
  } else if (selDet === "LHCB") {
    let ipx = 40, ipy = cy;
    let nTracks = 16;
    for (let i = 0; i < nTracks; i++) {
      let a = (Math.random() - 0.5) * 0.65;
      let len = 150 + Math.random() * 120;
      let curve = a + (Math.random() > 0.5 ? 0.08 : -0.08);
      let tx = ipx + len * Math.cos(a);
      let ty = ipy + len * Math.sin(a);
      let cpx = ipx + (len/2) * Math.cos(curve);
      let cpy = ipy + (len/2) * Math.sin(curve);
      let isMu = Math.random() > 0.85;
      ctxEv.strokeStyle = isMu ? "#2ea44f" : "#ff7f0e";
      ctxEv.lineWidth = isMu ? 1.5 : 0.8;
      ctxEv.beginPath(); ctxEv.moveTo(ipx, ipy); ctxEv.quadraticCurveTo(cpx, cpy, tx, ty); ctxEv.stroke();
      if (isMu && tx < w) {
        ctxEv.fillStyle = "#2ea44f";
        ctxEv.beginPath(); ctxEv.arc(tx, ty, 2.5, 0, 2*Math.PI); ctxEv.fill();
      }
    }
    ctxEv.fillStyle = "#ff7f0e"; ctxEv.font = "8px sans-serif";
    ctxEv.fillText("Forward-B-Zerfall", ipx + 90, ipy - 35);
  }
}

// ═══════════════════════════════════════════════════════════════════════════
// HISTOGRAM
// ═══════════════════════════════════════════════════════════════════════════
function generateMassData(){
 // Skalierungsfaktor basierend auf Intensität (quadratisch) und Squeeze (antiproportional)
 let rateFactor = Math.pow(paramIntensity, 2) / paramBetaStar;
 let nPointsMultiplier = Math.max(1, Math.round(rateFactor * 4));
 
 if(!isIon){
  // Background (5x weniger pro Kollision, da wir 5x mehr Kollisionen sammeln)
  let bgCount = Math.max(1, Math.round(4 * nPointsMultiplier));
  for(let i=0; i<bgCount; i++){
   let m=50+Math.exp(Math.random()*4.6)*1.5;
   if(m>50&&m<150)massData.push(m);
  }
  // Z0 Peak (immer vorhanden)
  let zCount = Math.max(1, Math.round(1.6 * nPointsMultiplier));
  for(let i=0; i<zCount; i++) {
   massData.push(91.2+(Math.random()-.5)*4+(Math.random()-.5)*2);
  }
  // Higgs Peak (Schwellenenergie-Kopplung! Wenn Energie < 4.0 TeV, wird kein Higgs erzeugt)
  if(paramEnergy >= 4.0) {
   let hCount = Math.max(1, Math.round(0.6 * nPointsMultiplier));
   for(let i=0; i<hCount; i++){
    let m=125+(Math.random()-.5)*2.8+(Math.random()-.5)*1.2;
    if(m>50&&m<150)massData.push(m);
   }
  }
 }else{
  // Heavy Ion: ALICE Quark-Gluon-Plasma Spektrum (5x weniger pro Kollision)
  let bgCount = Math.max(1, Math.round(6 * nPointsMultiplier));
  for(let i=0; i<bgCount; i++) massData.push(1+Math.random()*11);
  
  let jpsiCount = Math.max(1, Math.round(2.4 * nPointsMultiplier));
  for(let i=0; i<jpsiCount; i++) massData.push(3.1+(Math.random()-.5)*.4+(Math.random()-.5)*.2);
  
  let upsilonCount = Math.max(1, Math.round(0.8 * nPointsMultiplier));
  for(let i=0; i<upsilonCount; i++) massData.push(9.46+(Math.random()-.5)*.6+(Math.random()-.5)*.3);
 }
}

function getTargetDiscover() {
  if (isIon) return 300;
  if (selDet === "LHCB") return 400;
  return 500;
}

function getSignificance() {
  if (collisions === 0) return 0;
  if (!isIon && selDet !== "LHCB" && paramEnergy < 4.0) return 0;
  let target = getTargetDiscover();
  return 5.0 * Math.sqrt(collisions / target);
}

function drawHist(){
  let w=histW,h=histH;
  ctxHist.clearRect(0,0,w,h);
  ctxHist.strokeStyle="#30363d";ctxHist.lineWidth=1;
  ctxHist.beginPath();ctxHist.moveTo(30,8);ctxHist.lineTo(30,h-16);ctxHist.lineTo(w-8,h-16);ctxHist.stroke();
  ctxHist.fillStyle="#8b949e";ctxHist.font="7.5px sans-serif";
  let mn=isIon?1:50, mx=isIon?12:150;
  ctxHist.fillText(mn+" GeV",30,h-5);ctxHist.fillText(mx+" GeV",w-40,h-5);
  
  let sig = getSignificance();
  $("lbl-sig").innerText = sig.toFixed(2) + " σ";
  
  let sigBar = $("sig-bar");
  let sigStatus = $("lbl-sig-status");
  let target = getTargetDiscover();
  let progressPct = (paramEnergy < 4.0 && !isIon && selDet !== "LHCB") ? 0 : (collisions / target) * 100;
  let pct = Math.min(100, progressPct);
  sigBar.style.width = pct + "%";
  
  if (sig === 0) {
    sigStatus.innerText = "Rauschen (Kein Signal)";
    sigStatus.style.color = "#8b949e";
    sigBar.style.background = "#30363d";
  } else if (sig < 3.0) {
    sigStatus.innerText = "Rauschen (Keine Signifikanz)";
    sigStatus.style.color = "#8b949e";
    sigBar.style.background = "#58a6ff";
  } else if (sig < 5.0) {
    sigStatus.innerText = "⚠️ Signal-Hinweis (Evidence!)";
    sigStatus.style.color = "#ff7f0e";
    sigBar.style.background = "#ff7f0e";
  } else {
    sigStatus.innerText = "🌟 5σ ENTDECKUNG (Discovery!)";
    sigStatus.style.color = "#2ea44f";
    sigBar.style.background = "#2ea44f";
  }
  
  if(!massData.length){
    ctxHist.fillStyle="#8b949e";
    ctxHist.font="10px monospace";
    ctxHist.fillText("WARTEN AUF KOLLISIONSDATEN...",w/2-90,h/2);
    return;
  }
  
  let nb=40,bins=Array(nb).fill(0);
  massData.forEach(v=>{if(v>=mn&&v<mx){let i=Math.floor((v-mn)/(mx-mn)*nb);if(i>=0&&i<nb)bins[i]++;}});
  let maxB=Math.max(...bins,1),bw=(w-40)/nb;
  let fc=isIon?"rgba(227,119,194,0.4)":"rgba(88,166,255,0.4)";
  let tc=isIon?"#e377c2":"#58a6ff";
  for(let i=0;i<nb;i++){
    let bh=bins[i]/maxB*(h-30);let x=30+i*bw,y=h-16-bh;
    ctxHist.fillStyle=fc;ctxHist.fillRect(x,y,bw-1,bh);ctxHist.fillStyle=tc;ctxHist.fillRect(x,y,bw-1,1.5);
  }
  
  if (sig > 0) {
    let alpha = Math.min(1.0, collisions / target);
    ctxHist.save();
    ctxHist.globalAlpha = alpha;
    
    ctxHist.strokeStyle="rgba(248,81,73,1)";
    ctxHist.lineWidth=1.5;
    ctxHist.beginPath();
    for(let xp=30;xp<w-8;xp++){
      let v=mn+(xp-30)/(w-38)*(mx-mn),yv=0;
      if(!isIon){yv=Math.exp(-(v-50)/25)*18+(1/((v-91.2)**2+2.5**2))*300+(1/((v-125)**2+1.6**2))*10;}
      else{yv=4.2+(1/((v-3.1)**2+.15**2))*1.4+(1/((v-9.46)**2+.3**2))*.35;}
      let sc=isIon?6.5:20,yp=h-16-yv/sc*(h-35);yp=Math.max(8,Math.min(h-16,yp));
      xp===30?ctxHist.moveTo(xp,yp):ctxHist.lineTo(xp,yp);
    }
    ctxHist.stroke();
    
    ctxHist.fillStyle="#f0f6fc";
    ctxHist.font="8.5px sans-serif";
    if(!isIon){
      ctxHist.fillText("Z⁰ Boson (91 GeV)",w*.35,22);
      if(selDet === "LHCB") {
        ctxHist.fillText("B-Meson-Physik & CP-Asymmetrie (LHCb)!",w*.58,38);
      } else {
        if(paramEnergy >= 4.0) {
          ctxHist.fillText("Higgs-Boson (125 GeV) - 5σ Entdeckung!",w*.58,38);
        } else {
          ctxHist.fillStyle="rgba(248,81,73,0.6)";
          ctxHist.fillText("Higgs unterdrückt (E < 4 TeV)",w*.58,38);
        }
      }
    } else {
      ctxHist.fillText("J/Ψ (3.1 GeV) - Quark-Gluon-Plasma",w*.15,20);
      ctxHist.fillText("Υ (9.46 GeV)",w*.68,45);
    }
    ctxHist.restore();
  }
  
  if (sig < 5.0) {
    ctxHist.fillStyle="rgba(255,255,255,0.45)";
    ctxHist.font="8.5px monospace";
    if (sig === 0) {
      if (paramEnergy < 4.0 && !isIon && selDet !== "LHCB") {
        ctxHist.fillText("⚠️ Strahlenergie zu gering für Higgs-Produktion (< 4.0 TeV)!", w/2 - 145, 12);
      } else {
        ctxHist.fillText("Keine Kollisionen akkumuliert. Starte Kollisionen!", w/2 - 120, 12);
      }
    } else {
      ctxHist.fillText("Sammle Statistik für Entdeckung (Signifikanz: " + sig.toFixed(1) + "σ / 5.0σ)", w/2 - 140, 12);
    }
  }
}

// ═══════════════════════════════════════════════════════════════════════════
// BUTTON HANDLERS
// ═══════════════════════════════════════════════════════════════════════════
btnB1.addEventListener("click",()=>injectBunch(1));
btnB2.addEventListener("click",()=>injectBunch(2));

btnSpeedToggle.addEventListener("click",()=>{
  isFastMode = !isFastMode;
  if(isFastMode) {
    btnSpeedToggle.innerText = "⏱️ Modus: Zeitraffer (Schnell)";
    btnSpeedToggle.style.background = "rgba(88,166,255,.08)";
    btnSpeedToggle.style.borderColor = "rgba(88,166,255,.3)";
    btnSpeedToggle.style.color = "#58a6ff";
  } else {
    btnSpeedToggle.innerText = "⏱️ Modus: Echtzeit (Didaktisch)";
    btnSpeedToggle.style.background = "rgba(227,119,194,.08)";
    btnSpeedToggle.style.borderColor = "rgba(227,119,194,.3)";
    btnSpeedToggle.style.color = "#e377c2";
  }
});

// GEO OVERLAY TOGGLE
let geoVisible = true;
btnToggleGeo.addEventListener("click",()=>{
 geoVisible = !geoVisible;
 document.querySelectorAll(".geo-element").forEach(el=>{
  el.style.opacity = geoVisible ? "1" : "0";
  el.style.pointerEvents = geoVisible ? "auto" : "none";
 });
 btnToggleGeo.classList.toggle("act", geoVisible);
});

// INTERACTIVE SVG CAMERA ZOOM LOGIC
let currentVB = {x:0, y:0, w:700, h:480};
let zoomTarget = null; // "ATLAS", "CMS", "ALICE", "LHCB" or null

function animateViewBox(tx, ty, tw, th, dur=500){
 const startX = currentVB.x, startY = currentVB.y, startW = currentVB.w, startH = currentVB.h;
 let t0 = null;
 function step(ts){
  if(!t0) t0 = ts;
  let p = Math.min((ts-t0)/dur, 1);
  let ep = p*p*(3-2*p); // Custom cubic ease-in-out
  currentVB.x = startX + ep*(tx-startX);
  currentVB.y = startY + ep*(ty-startY);
  currentVB.w = startW + ep*(tw-startW);
  currentVB.h = startH + ep*(th-startH);
  svg.setAttribute("viewBox", `${currentVB.x} ${currentVB.y} ${currentVB.w} ${currentVB.h}`);
  if(p<1) requestAnimationFrame(step);
 }
 requestAnimationFrame(step);
}

function zoomToDetector(name){
 if(zoomTarget === name){
  // Zoom out
  zoomTarget = null;
  btnZoomOut.classList.add("off");
  animateViewBox(0, 0, 700, 480);
 } else {
  zoomTarget = name;
  btnZoomOut.classList.remove("off");
  let tx, ty, tw=160, th=120;
  if(name === "ATLAS") { tx = 270; ty = 360; }
  else if(name === "CMS") { tx = 270; ty = 0; }
  else if(name === "ALICE") { tx = 90; ty = 180; }
  else if(name === "LHCB") { tx = 450; ty = 180; }
  animateViewBox(tx, ty, tw, th);
  
  // Update event display tabs too
  document.querySelectorAll(".cv4-dtab").forEach(t=>t.classList.remove("act"));
  $("dt-"+name.toLowerCase()).classList.add("act");
  selDet=name;
  drawDetBg();
 }
}

btnZoomOut.addEventListener("click", () => {
 zoomTarget = null;
 btnZoomOut.classList.add("off");
 animateViewBox(0, 0, 700, 480);
});

grpAtlas.addEventListener("click", () => zoomToDetector("ATLAS"));
grpCms.addEventListener("click", () => zoomToDetector("CMS"));
grpAlice.addEventListener("click", () => zoomToDetector("ALICE"));
grpLhcb.addEventListener("click", () => zoomToDetector("LHCB"));

// PRESETS CLICK LISTENERS
btnPreHiggs.addEventListener("click",()=>{
 setMode(false); // Protonen
 resetLHC();
 sliEnergy.value = 6.8; paramEnergy = 6.8; lblEnergy.innerText = "6.8 TeV";
 sliIntensity.value = 1.20; paramIntensity = 1.20; lblIntensity.innerText = "1.20e11 p";
 sliBeta.value = 0.3; paramBetaStar = 0.3; lblBeta.innerText = "0.30 m";
 sliRampSpeed.value = 0.05; paramRampSpeed = 0.05; lblRampSpeed.innerText = "0.05 T/s (Sicher)"; lblRampSpeed.style.color = "#58a6ff";
 document.querySelectorAll(".cv4-dtab").forEach(t=>t.classList.remove("act"));
 $("dt-atlas").classList.add("act"); selDet="ATLAS";
 updateReadouts(); drawDetBg(); drawHist();
 setStatus("PRESET GELADEN: Higgs-Boson-Suche (Standardmodell p-p Kollision bei 13.6 TeV)", "on");
});

btnPreQgp.addEventListener("click",()=>{
 setMode(true); // Blei-Ionen
 resetLHC();
 sliEnergy.value = 2.5; paramEnergy = 2.5; lblEnergy.innerText = "2.5 TeV";
 sliIntensity.value = 0.90; paramIntensity = 0.90; lblIntensity.innerText = "0.90e11 p";
 sliBeta.value = 0.4; paramBetaStar = 0.4; lblBeta.innerText = "0.40 m";
 sliRampSpeed.value = 0.05; paramRampSpeed = 0.05; lblRampSpeed.innerText = "0.05 T/s (Sicher)"; lblRampSpeed.style.color = "#58a6ff";
 document.querySelectorAll(".cv4-dtab").forEach(t=>t.classList.remove("act"));
 $("dt-alice").classList.add("act"); selDet="ALICE";
 updateReadouts(); drawDetBg(); drawHist();
 setStatus("PRESET GELADEN: Blei-Ionen-Kollision zur Erzeugung des Quark-Gluon-Plasmas in ALICE", "on");
});

btnPreLhcb.addEventListener("click",()=>{
 setMode(false); // Protonen
 resetLHC();
 sliEnergy.value = 6.5; paramEnergy = 6.5; lblEnergy.innerText = "6.5 TeV";
 sliIntensity.value = 1.00; paramIntensity = 1.00; lblIntensity.innerText = "1.00e11 p";
 sliBeta.value = 0.6; paramBetaStar = 0.6; lblBeta.innerText = "0.60 m";
 sliRampSpeed.value = 0.05; paramRampSpeed = 0.05; lblRampSpeed.innerText = "0.05 T/s (Sicher)"; lblRampSpeed.style.color = "#58a6ff";
 document.querySelectorAll(".cv4-dtab").forEach(t=>t.classList.remove("act"));
 $("dt-lhcb").classList.add("act"); selDet="LHCB";
 updateReadouts(); drawDetBg(); drawHist();
 setStatus("PRESET GELADEN: CP-Verletzung & Schönheit (B-Physik p-p Kollision bei 13 TeV in LHCb)", "on");
});

btnPrePilot.addEventListener("click",()=>{
 setMode(false); // Protonen
 resetLHC();
 sliEnergy.value = 0.4; paramEnergy = 0.4; lblEnergy.innerText = "0.4 TeV";
 sliIntensity.value = 0.10; paramIntensity = 0.10; lblIntensity.innerText = "0.10e11 p";
 sliBeta.value = 1.5; paramBetaStar = 1.5; lblBeta.innerText = "1.50 m";
 sliRampSpeed.value = 0.02; paramRampSpeed = 0.02; lblRampSpeed.innerText = "0.02 T/s (Sicher)"; lblRampSpeed.style.color = "#58a6ff";
 updateReadouts(); drawDetBg(); drawHist();
 setStatus("PRESET GELADEN: Pilot-Strahl (Inbetriebnahme des LHC auf Injektionsniveau)", "on");
});

btnAutoColl.addEventListener("click", toggleAutoCollide);

btnAuto.addEventListener("click",async()=>{
 if(injecting||ramped||b1Count>=NEEDED||b2Count>=NEEDED||cryoRecovery) return;
 btnAuto.classList.add("off");
 btnB1.classList.add("off");
 btnB2.classList.add("off");
 sliEnergy.disabled = true; // Sperre Parameter während Injektion
 
 // Realistischer Beladungsprozess: Zuerst Beam 1 komplett füllen, danach erst Beam 2!
 setStatus("FÜLLPROTOKOLL GESTARTET: Fülle zuerst Beam 1 (im Uhrzeigersinn)...","on");
 
 for(let i=b1Count; i<NEEDED; i++){
  setStatus(`PROTOKOLL: Fülle LHC Beam 1 (Clockwise) - Bunch ${i+1}/${NEEDED}...`,"on");
  await injectBunch(1);
  if(i < NEEDED - 1) {
    await new Promise(r=>setTimeout(r, getDurations().autoDelay));
  }
 }
 
 await new Promise(r=>setTimeout(r, getDurations().autoDelay * 2));
 setStatus("PROTOKOLL: Beam 1 stabil! Fülle nun Beam 2 (gegen den Uhrzeigersinn)...","on");
 
 for(let i=b2Count; i<NEEDED; i++){
  setStatus(`PROTOKOLL: Fülle LHC Beam 2 (Counter-Clockwise) - Bunch ${i+1}/${NEEDED}...`,"on");
  await injectBunch(2);
  if(i < NEEDED - 1) {
    await new Promise(r=>setTimeout(r, getDurations().autoDelay));
  }
 }
 
 setStatus("FÜLLSCHEMA BEENDET: Beide Strahlen unabhängig gefüllt und stabil! Ramping bereit.","on");
 btnAuto.classList.remove("off");
});

// SLIDERS BINDING
sliEnergy.addEventListener("input",()=>{
 paramEnergy = parseFloat(sliEnergy.value);
 lblEnergy.innerText = paramEnergy.toFixed(1) + " TeV";
 updateReadouts();
});

sliIntensity.addEventListener("input",()=>{
 paramIntensity = parseFloat(sliIntensity.value);
 lblIntensity.innerText = paramIntensity.toFixed(2) + "e11 p";
});

sliBeta.addEventListener("input",()=>{
 lblBeta.innerText = parseFloat(sliBeta.value).toFixed(2) + " m";
});

sliRampSpeed.addEventListener("input",()=>{
 paramRampSpeed = parseFloat(sliRampSpeed.value);
 if(paramRampSpeed > 0.10) {
  lblRampSpeed.innerText = paramRampSpeed.toFixed(2) + " T/s (⚠️ RISIKO)";
  lblRampSpeed.style.color = "#f85149";
 } else {
  lblRampSpeed.innerText = paramRampSpeed.toFixed(2) + " T/s (Sicher)";
  lblRampSpeed.style.color = "#58a6ff";
 }
});

// ═══════════════════════════════════════════════════════════════════════════
// INIT
// ═══════════════════════════════════════════════════════════════════════════
resizeCanvases();
updateReadouts(); drawDetBg(); drawHist();
setStatus("BEREIT — Wähle Teilchenart und starte Injektion","on");

// Handle resize dynamically
window.addEventListener("resize", ()=>{
 resizeCanvases(); drawDetBg(); drawHist();
});

})();
</script>'''))


## 📈 Post-Kollisions-Analyse

Nach zahlreichen Kollisionen analysieren wir das akkumulierte Massenspektrum.
Setze `heavy_ion_analysis = True` für Blei-Ionen (J/ψ, Υ) oder `False` für Protonen (Higgs, Z0).

> [!TIP]
> **Didaktischer Hinweis zur Entdeckung & Statistik**:
> Im realen LHC ist das Signal-zu-Rausch-Verhältnis (Signal-to-Background Ratio) für das Higgs-Boson extrem winzig (z.B. ca. 1 zu 1000 im Zerfallskanal H → ZZ* → 4l). Um die Entdeckung in unserem Schul-Bedienfeld innerhalb weniger Sekunden erlebbar zu machen, wurde die statistische Higgs-Kopplungsstärke didaktisch vergrößert. Die Notwendigkeit, eine signifikante Anzahl an Kollisionen (Statistik) zu sammeln, um das Signal über das Rauschen (Fluktuationen) zu heben, bleibt jedoch vollkommen identisch!


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.special import voigt_profile

# ── Betriebsmodus wählen: "HIGGS" | "LHCB" | "QGP" | "PILOT" ──
# (entspricht der globalen Statusvariable activePhysicsMode der Schaltzentrale)
MODE        = "HIGGS"
BEAM_ENERGY = 6.8   # TeV – steuert die Higgs-Schwellenkopplung (E >= 4.0 TeV)

try: cu.apply_cern_style()
except: plt.style.use('dark_background')

MODES = {
    "HIGGS": dict(xlim=(50, 150),   bins=80, color="#58a6ff", einheit="GeV/c²",
                  titel="⚛️ p-p: Z⁰ (Voigt) & Higgs-Boson"),
    "LHCB":  dict(xlim=(4.5, 6.0),  bins=70, color="#ff7f0e", einheit="GeV/c²",
                  titel="⚛️ LHCb: CP-Asymmetrie B⁰ / anti-B⁰"),
    "QGP":   dict(xlim=(1.0, 12.0), bins=70, color="#e377c2", einheit="GeV/c²",
                  titel="⚛️ Pb-Pb: Quark-Gluon-Plasma (J/ψ, Υ)"),
    "PILOT": dict(xlim=(0.4, 2.0),  bins=60, color="#ffd700", einheit="GeV/c²",
                  titel="⚛️ Pilot-Lauf: ρ⁰/ω & φ (unkalibriert)"),
}
cfg    = MODES[MODE]
lo, hi = cfg["xlim"]
rng    = np.random.default_rng(42)
fig, ax = plt.subplots(figsize=(12, 7))

if MODE == "HIGGS":
    bg = rng.exponential(35, 14000) + 50
    z  = 91.18 + (2.49/2) * rng.standard_cauchy(6000)   # Breit-Wigner (Γ = 2.49 GeV)
    z  = z + rng.normal(0, 1.6, z.size)                 # ⊗ Detektorauflösung (σ = 1.6)
    parts, higgs_on = [bg, z], BEAM_ENERGY >= 4.0
    if higgs_on:
        parts.append(rng.normal(125.0, 1.6, 450))
    data = np.concatenate(parts); data = data[(data >= lo) & (data <= hi)]
    h, e = np.histogram(data, bins=cfg["bins"], range=(lo, hi)); c = .5*(e[1:]+e[:-1])

    def model(x, a_bg, lam, a_z, m_z, a_h, m_h):
        return (a_bg*np.exp(-(x-50)/lam)
                + a_z*voigt_profile(x - m_z, 1.6, 2.49/2)      # Voigt = BW ⊗ Gauß
                + a_h*np.exp(-.5*((x - m_h)/1.6)**2))
    p, pc = curve_fit(model, c, h, p0=[500, 30, 800, 91.2, 200, 125],
                      bounds=([0, 5, 0, 88, 0, 120], [5e3, 200, 5e4, 94, 5e3, 130]),
                      maxfev=20000)
    perr = np.sqrt(np.diag(pc)); xf = np.linspace(lo, hi, 1000)
    ax.errorbar(c, h, yerr=np.sqrt(h), fmt='o', color='#8b949e', ms=3, label='p-p Daten')
    ax.plot(xf, model(xf, *p), color=cfg["color"], lw=2.5, label='Fit (Voigt-Z⁰ + Higgs)')
    hl = (f'$M(H) = {p[5]:.2f}\\pm{perr[5]:.2f}$ GeV' if higgs_on
          else 'Higgs unterdrückt (E < 4 TeV)')
    txt = f'$M(Z^0) = {p[3]:.2f}\\pm{perr[3]:.2f}$ GeV (Voigt)\n{hl}'

elif MODE == "LHCB":
    # CPT: identische Masse für B⁰ und anti-B⁰; CP-Verletzung -> ~15% Ratenasymmetrie
    def lhcb_set(n_sig):
        d = np.concatenate([rng.uniform(lo, hi, 4000), rng.normal(5.28, 0.05, n_sig)])
        return d[(d >= lo) & (d <= hi)]
    h_m, e = np.histogram(lhcb_set(1500),            bins=cfg["bins"], range=(lo, hi))
    h_a, _ = np.histogram(lhcb_set(int(1500*0.85)),  bins=cfg["bins"], range=(lo, hi))
    c = .5*(e[1:]+e[:-1])
    def model(x, a_bg, a, m, s): return a_bg + a*np.exp(-.5*((x-m)/s)**2)
    bnds = ([0, 0, 5.1, 0.01], [1e3, 1e4, 5.45, 0.2])
    p_m, _ = curve_fit(model, c, h_m, p0=[50, 300, 5.28, 0.05], bounds=bnds, maxfev=20000)
    p_a, _ = curve_fit(model, c, h_a, p0=[50, 250, 5.28, 0.05], bounds=bnds, maxfev=20000)
    xf = np.linspace(lo, hi, 1000)
    ax.step(c, h_m, where='mid', color='#58a6ff', alpha=.45, label='B⁰ → K⁺π⁻ (Materie)')
    ax.step(c, h_a, where='mid', color='#f85149', alpha=.45, label='anti-B⁰ → K⁻π⁺ (Antimaterie)')
    ax.plot(xf, model(xf, *p_m), color='#58a6ff', lw=2.5)
    ax.plot(xf, model(xf, *p_a), color='#f85149', lw=2.5)
    a_cp = (p_m[1] - p_a[1]) / (p_m[1] + p_a[1])
    txt = f'$M(B^0) = {p_m[2]:.3f}$ GeV (CPT-konform)\nCP-Asymmetrie $A_{{CP}} = {a_cp:.1%}$'

elif MODE == "QGP":
    bg = rng.uniform(lo, hi, 9750)
    jp = rng.normal(3.10, 0.15, 4050)
    up = rng.normal(9.46, 0.30, 1200)
    data = np.concatenate([bg, jp, up]); data = data[(data >= lo) & (data <= hi)]
    h, e = np.histogram(data, bins=cfg["bins"], range=(lo, hi)); c = .5*(e[1:]+e[:-1])
    def model(x, a_bg, aj, mj, sj, au, mu, su):
        return a_bg + aj*np.exp(-.5*((x-mj)/sj)**2) + au*np.exp(-.5*((x-mu)/su)**2)
    p, pc = curve_fit(model, c, h, p0=[150, 200, 3.1, 0.15, 50, 9.46, 0.30],
                      bounds=([0, 0, 2.8, 0.05, 0, 9.0, 0.1], [1e3, 1e4, 3.4, 0.4, 1e4, 9.9, 0.6]),
                      maxfev=20000)
    xf = np.linspace(lo, hi, 1000)
    ax.errorbar(c, h, yerr=np.sqrt(h), fmt='o', color='#8b949e', ms=3, label='Pb-Pb Daten (ALICE)')
    ax.plot(xf, model(xf, *p), color=cfg["color"], lw=2.5, label='Fit')
    txt = f'$M(J/\\psi) = {p[2]:.3f}$ GeV\n$M(\\Upsilon) = {p[5]:.3f}$ GeV'

else:  # PILOT – niedrige Intensität, unkalibriert, stark verrauscht
    bg  = rng.exponential(0.6, 8000) + 0.4
    rho = rng.normal(0.78, 0.10, 900)
    phi = rng.normal(1.02, 0.06, 500)
    data = np.concatenate([bg, rho, phi]); data = data[(data >= lo) & (data <= hi)]
    h, e = np.histogram(data, bins=cfg["bins"], range=(lo, hi)); c = .5*(e[1:]+e[:-1])
    h = np.maximum(h + rng.normal(0, np.sqrt(np.maximum(h, 1))*1.5, h.size), 0)  # induziertes Rauschen
    ax.errorbar(c, h, yerr=np.sqrt(np.maximum(h, 1)), fmt='o', color='#8b949e', ms=3,
                label='Teststrahl (verrauscht)')
    def model(x, a_bg, lam, ar, sr, ap, sp):
        return (a_bg*np.exp(-x/lam) + ar*np.exp(-.5*((x-0.78)/sr)**2)
                + ap*np.exp(-.5*((x-1.02)/sp)**2))
    try:
        p, _ = curve_fit(model, c, h, p0=[50, 0.6, 80, 0.1, 60, 0.06],
                         bounds=([0, 0.1, 0, 0.04, 0, 0.02], [1e3, 3, 1e4, 0.3, 1e4, 0.2]),
                         maxfev=20000)
        ax.plot(np.linspace(lo, hi, 1000), model(np.linspace(lo, hi, 1000), *p),
                color=cfg["color"], lw=2, label='Tentativer Fit')
    except RuntimeError:
        pass
    txt = 'ρ⁰/ω (0.78) & φ (1.02 GeV)\n⚠️ Unkalibriert – Fit instabil'

ax.set_title(cfg["titel"], fontsize=14, fontweight='bold', color=cfg["color"])
ax.set_xlabel(f'Invariante Masse [{cfg["einheit"]}]'); ax.set_ylabel('Ereignisse')
ax.set_xlim(lo, hi)
ax.text(.05, .95, txt, transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', fc='#161b22', ec='#30363d', alpha=.85), color='#fff')
ax.legend(framealpha=.8, loc='upper right')
plt.tight_layout(); plt.show()


## 📊 Statistik, Signifikanz & Maschinenphysik

Eine Entdeckung ist erst ab **5σ** anerkannt. Die Signifikanz wächst mit der gesammelten
integrierten Luminosität gemäß der statistischen Quadratwurzel-Regel `Z = 1.58·√L_int`
(5σ exakt bei 10 fb⁻¹ – der reale Higgs-Meilenstein vom 4. Juli 2012). Die Datenrate
selbst skaliert mit der Strahlphysik (`Intensität² / β*`). Im `PILOT`-Modus ist sie um
95 % gedrosselt, sodass die Entdeckungsgrenze realistischerweise nie erreicht wird.

Für Blei-Ionen (`QGP`) begrenzt die Supraleitungsgrenze der Dipolmagnete die Strahlenergie
auf **2.56 TeV/Nukleon**; B-Feld und Lorentz-γ folgen dem Ladungs-zu-Masse-Verhältnis von Pb-208.

In [ ]:
# ── Luminositäts-Akkumulation & Entdeckungs-Signifikanz ──
# Datenrate proportional zur Strahlphysik:   dL_int ∝ Intensität² / β*
# Signifikanz folgt der statistischen √-Regel: Z = 1.58 · √L_int
#   → 5σ exakt bei L_int = 10 fb⁻¹ (Higgs-Entdeckung, 4. Juli 2012)
import numpy as np, matplotlib.pyplot as plt
try: cu.apply_cern_style()
except: plt.style.use('dark_background')

INTENSITY = 1.15   # 1e11 Protonen pro Bunch
BETA_STAR = 0.30   # m  (Squeeze am Kollisionspunkt)

steps = np.arange(0, 140)
dL    = 0.15 * INTENSITY**2 / BETA_STAR
rate  = dL * (0.05 if MODE == "PILOT" else 1.0)   # Pilot-Lauf: 95% gedrosselt
L_int = rate * steps
Z     = 1.58 * np.sqrt(L_int)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(L_int, Z, color='#58a6ff', lw=2.5, label=f'Z = 1.58·√L_int   (Modus: {MODE})')
ax.axhline(3.0, color='#ff7f0e', ls='--', lw=1.2, label='3σ – Evidence')
ax.axhline(5.0, color='#2ea44f', ls='--', lw=1.5, label='5σ – Discovery')
ax.axvline(10.0, color='#2ea44f', ls=':', lw=1, alpha=.6)
ax.annotate('4. Juli 2012\n10 fb⁻¹ → 5σ', xy=(10, 5.0), xytext=(12, 3.3),
            color='#2ea44f', fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#2ea44f'))
ax.set_xlabel('Integrierte Luminosität $L_{int}$ [fb⁻¹]')
ax.set_ylabel('Signifikanz Z [σ]')
ax.set_title('📊 Statistik-Akkumulation: Vom Rauschen zur Entdeckung',
             fontsize=14, fontweight='bold', color='#58a6ff')
ax.set_xlim(0, max(L_int.max(), 12)); ax.set_ylim(0, max(Z.max(), 6))
ax.legend(framealpha=.8, loc='lower right')
plt.tight_layout(); plt.show()

print(f'Modus {MODE}: Rate = {rate:.4f} fb⁻¹/Schritt  →  Z nach {steps[-1]} Schritten = {Z[-1]:.2f} σ')
if MODE == "PILOT":
    print('⚠️ Pilot-Lauf: Luminosität um 95% gedrosselt – 5σ wird im Normalbetrieb nie erreicht.')

# ── Kinematik-Korrekturen für Blei-Ionen (Pb⁸²⁺, A=208, Z=82) ──
def pb_ion_kinematik(E_nucleon_TeV):
    """Dipolfeld B [T] und Lorentz-γ pro Nukleon; Energie auf 2.56 TeV/u begrenzt."""
    E_cap = min(E_nucleon_TeV, 2.56)            # Supraleitungsgrenze der Magnete (~8.33 T)
    E_gev = E_cap * 1000.0
    B     = (208/82) * E_gev / (0.299792458 * 2803.95)   # Magnetische Rigidität (Z/A)
    gamma = E_gev / 0.9315                       # E_nucleon / m_Nukleon [GeV]
    return E_cap, B, gamma

E_cap, B, g = pb_ion_kinematik(2.56)
print(f'\nPb⁸²⁺ @ {E_cap:.2f} TeV/u (Maximum):  B = {B:.3f} T   γ = {g:,.0f}')


## 🎓 Zusammenfassung

In dieser Simulation hat **dasselbe Teilchenpaket** den gesamten Beschleunigerkomplex durchlaufen:
- Vom Injektor (LINAC 4/3) durch die Vorbeschleuniger (PSB/LEIR → PS → SPS) bis zur Einspeisung in den LHC.
- Der LHC wurde tatsächlich aufgefüllt: Jeder Injektionszyklus hat ein weiteres Bunch zum Strahl hinzugefügt.
- Erst bei ausreichender Füllung beider Strahlen konnte die Energie gerampt und Kollisionen ausgelöst werden.
- Die Pfade im Stellwerk sind geometrisch exakt verbunden (Transferlinien TI 2 und TI 8 verbinden SPS mit LHC an den korrekten Einspeisepunkten nahe ALICE und LHCb).
